# ASGC on Transformers - CUDA GPU Validation Notebook

**Question:** Does the Adaptive Staleness-Aware Gradient Compression (ASGC) framework work on transformer models, and does the adaptive controller deliver measurable gains under worker heterogeneity?

This notebook trains a GPT-style causal transformer on Tiny Shakespeare through `torch_asgc`'s asynchronous parameter server. It **requires a CUDA GPU** by default: if `torch.cuda.is_available()` is false the setup cell stops with an explicit error instead of silently falling back to CPU. Set `ASGC_DISABLE_CUDA=1` only for local debugging or a CPU-only smoke test.

### What is measured
- **Wall-clock** to a fixed accepted-update budget (async vs barrier-synchronous).
- **Uplink and downlink wire bytes** using the library's exact compressed-payload accounting.
- **Validation loss and perplexity** on a held-out split.
- **Top-1 and top-5 next-token accuracy** on the held-out split. The random baseline is about 1/65 (1.5%) because the character vocabulary has 65 symbols.
- **Staleness, rejection, and downweight rates**.
- **Adaptive controller trajectory** (`rho` and `S_max`).
- **Time-to-target loss**, with the target derived from the uncompressed async arm.

### Experimental arms
| arm | compression | error feedback | controller | synchronization |
|---|---|---|---|---|
| `async_identity` | none | no | no | async |
| `async_topk20` | Top-K 20% | yes | no | async |
| `async_adaptive` | Top-K, `rho` in [0.10, 0.20] | yes | yes | async |
| `sync_identity` | none | no | no | barrier-synchronous |

### Why 20% Top-K, not 50%
Top-K stores an `int32` index plus a `float32` value per selected entry: **8 bytes per entry**, while dense FP32 is 4 bytes per parameter. 50% Top-K is therefore byte-neutral. 20% Top-K gives roughly a **2.5x uplink reduction** before any adaptive savings.

### Hardware strategy on a single CUDA GPU
The notebook assumes **one visible CUDA device** (the common Colab/Kaggle GPU case). ASGC workers run concurrently, so only one worker may own the GPU. The notebook therefore:
- runs **worker 0 on the CUDA GPU** when preflight succeeds,
- runs the **parameter server and workers 1-3 on CPU**,
- runs the **final evaluation on the GPU** after workers stop,
- moves all compressed payloads to CPU before the parameter-server thread handles them.

Worker heterogeneity is injected with per-worker sleep profiles (base delay + jitter + periodic spikes) and a simulated per-worker uplink/downlink bandwidth. On a GPU the fastest worker is pinned to the device and the CPU workers act as stragglers, which is intentional: the experiment is about staleness under heterogeneous compute, not about maximizing GPU utilization.

## Before you run

1. Select a **GPU runtime** (Colab: *Runtime > Change runtime type > T4/L4/A100 GPU*; Kaggle: *Accelerator > GPU T4 x2 / P100 / L4*). Verify `torch.cuda.is_available()` is `True`.
2. Make the ASGC repository available. The setup cell searches common paths and clones `https://github.com/ksacheth/parallel-computing.git` when needed. To use an uploaded copy, set `ASGC_REPO=/path/to/parallel-computing`.
3. Run all cells. The default profile is `report` with **one seed** (`0`) and roughly 5-10 minutes on a single GPU. For a smoke test, set `ASGC_NOTEBOOK_PROFILE=quick` before starting the kernel, or edit `PROFILE` in the configuration cell.
4. No automatic `pip install` is performed. `torch`, `matplotlib`, and `numpy` should already be present in the runtime. The notebook prints a dependency report and continues without plots if `matplotlib` is missing. A CUDA-enabled PyTorch wheel is required; a CPU-only wheel cannot see the GPU.

Useful environment variables:
- `ASGC_NOTEBOOK_PROFILE=quick|report`
- `ASGC_SEEDS=0`
- `ASGC_ARMS=async_identity,async_topk20,async_adaptive,sync_identity`
- `ASGC_OUT=/content/asgc_transformer_results`
- `ASGC_CUDA_DEVICE=0` (which visible GPU worker 0 uses)
- `ASGC_DISABLE_CUDA=1` (force CPU-only for local debugging; also disables the GPU requirement)
- `ASGC_CUDA_STRICT=0` (run on CPU instead of failing when CUDA is unavailable)
- `ASGC_TIME_BUDGET=900` (stop launching new arms after N seconds)
- `ASGC_TARGET`, `ASGC_BANDWIDTH`, `ASGC_DOWNLINK`, `ASGC_RHO_MIN`, `ASGC_RHO_MAX`, `ASGC_SMAX_MIN`, `ASGC_SMAX_MAX`, `ASGC_NUM_WORKERS`

### If CUDA is unavailable

Check the runtime and the installed wheel:

```python
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda build:", torch.version.cuda)
print("device count:", torch.cuda.device_count())
```

If `torch.cuda.is_available()` is `False` on a GPU runtime, the installed PyTorch is a CPU-only build. Install a CUDA build matching the runtime (the default PyPI wheel for your platform usually works), restart the kernel, and rerun. The raised error also prints the detected versions. Set `ASGC_DISABLE_CUDA=1` to run the whole experiment on CPU instead.

In [ ]:
from __future__ import annotations

import copy
import csv
import gc
import json
import math
import os
import platform
import random
import shutil
import statistics
import subprocess
import sys
import threading
import time
import urllib.request
import warnings
import zipfile
from dataclasses import asdict, dataclass, replace
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

if os.environ.get("ASGC_DISABLE_CUDA") != "1":
    os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")

REPO_URL = "https://github.com/ksacheth/parallel-computing.git"


def _looks_like_repo(path: Path) -> bool:
    return (path / "src" / "torch_asgc" / "__init__.py").is_file()


def find_repo() -> Path:
    candidates: list[Path] = []
    env_repo = os.environ.get("ASGC_REPO")
    if env_repo:
        candidates.append(Path(env_repo).expanduser())
    candidates += [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/parallel-computing"),
        Path("/kaggle/working/parallel-computing"),
        Path.cwd() / "parallel-computing",
        Path.home() / "parallel-computing",
    ]
    seen: set[Path] = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if _looks_like_repo(resolved):
            return resolved

    clone_dir = Path.cwd() / "parallel-computing"
    if not clone_dir.exists():
        print(f"ASGC repo not found; cloning {REPO_URL} -> {clone_dir}")
        try:
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)],
                check=True,
                timeout=180,
            )
        except Exception as exc:
            raise RuntimeError(
                "Could not locate or clone the ASGC repository. "
                "Set ASGC_REPO to the repository path, or upload/clone it manually."
            ) from exc

    if _looks_like_repo(clone_dir):
        return clone_dir.resolve()
    for child in Path.cwd().iterdir():
        if child.is_dir() and _looks_like_repo(child):
            return child.resolve()
    raise RuntimeError("Repository found but src/torch_asgc is missing. Check ASGC_REPO.")


REPO_DIR = find_repo()
SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("repo:", REPO_DIR)
print("cwd:", Path.cwd())
print("python:", sys.version.split()[0])
print("torch:", __import__("torch").__version__)
print("cuda:", end=" ")
try:
    import torch as _torch
    print(
        f"available={_torch.cuda.is_available()} "
        f"build={_torch.version.cuda} count={_torch.cuda.device_count()}"
    )
    if _torch.cuda.is_available():
        print("gpu:", _torch.cuda.get_device_name(0))
except Exception as exc:
    print(f"not available ({type(exc).__name__}: {exc})")
for _name in ("numpy", "matplotlib", "pandas", "IPython"):
    try:
        _module = __import__(_name)
        print(f"{_name}: {getattr(_module, '__version__', 'installed')}")
    except Exception:
        print(f"{_name}: not installed")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch_asgc
from torch_asgc import (
    ASGCClient,
    AdaptiveConfig,
    CompressionConfig,
    DistributedModel,
    DistributedOptimizer,
    LocalTransportHub,
    ParameterServer,
    StalenessConfig,
    connect,
)
from torch_asgc.compression import DensePiece, QuantizedPiece, SparsePiece
from torch_asgc.transport.base import (
    BucketPayload,
    FetchRequest,
    FetchResponse,
    PushAck,
    PushUpdate,
    TransportClient,
)
from torch_asgc.transport.local import LocalTransportClient

try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass
CPU_THREADS = max(1, int(os.environ.get("ASGC_CPU_THREADS", "2")))
torch.set_num_threads(CPU_THREADS)

print("torch_asgc:", getattr(torch_asgc, "__version__", "unknown"))
print("library path:", Path(torch_asgc.__file__).parent)
print("torch threads:", CPU_THREADS)

## Configuration

`report` is the profile intended for the findings. `quick` is a smoke test that exercises every code path in a couple of minutes.

The adaptive arm starts at the same compression level as the fixed Top-K baseline (`rho = 0.20`). Its controller may move `rho` in either direction within `[0.10, 0.20]` (steps of 0.05) and `S_max` within `[6, 12]` (steps of 1): high communication pressure with stable loss drives `rho` down and `S_max` up, while instability or worsening loss drives `rho` back up and `S_max` down. Because it starts at the same 20% active fraction, any byte saving is earned by reacting to telemetry rather than by starting from a different compression level.

In [ ]:
@dataclass
class Arm:
    name: str
    mode: str
    rho: float
    error_feedback: bool
    adaptive: bool
    sync: bool
    color: str


@dataclass
class ExpConfig:
    seed: int = 0
    profile_name: str = "report"
    num_workers: int = 4
    target_updates: int = 200
    time_cap_s: float = 420.0
    eval_every: int = 20
    eval_interval_s: float = 3600.0  # time fallback; short runs use the version grid
    eval_batches: int = 10
    batch_size: int = 8
    block_size: int = 128
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 192
    dropout: float = 0.1
    lr: float = 5e-4
    weight_decay: float = 0.1
    bandwidth_mbps: float = 32.0
    downlink_mbps: float = 128.0
    latency_s: float = 0.002
    bucket_cap_mb: float = 25.0
    local_steps: int = 1
    max_grad_norm: float | None = 1.0
    s_low: int = 2
    s_max: int = 10
    beta: float = 0.5
    adaptive_interval_s: float = 2.0
    rho_min: float = 0.10
    rho_max: float = 0.20
    max_delta_rho: float = 0.05
    smax_min: int = 6
    smax_max: int = 12
    max_delta_smax: int = 1
    comm_pressure_min: float = 200_000.0
    worsen_tol: float = 0.05
    variability_tol: float = 0.05
    reject_frac_max: float = 0.25
    use_gpu_worker: bool = True
    eval_on_gpu: bool = True
    cuda_strict: bool = True


WORKER_PROFILES = [
    {"base": 0.000, "jitter": 0.004, "spike_every": 9, "spike": 0.10},
    {"base": 0.050, "jitter": 0.008, "spike_every": 7, "spike": 0.25},
    {"base": 0.200, "jitter": 0.012, "spike_every": 11, "spike": 0.50},
    {"base": 0.700, "jitter": 0.020, "spike_every": 5, "spike": 1.10},
]


def make_arms() -> list[Arm]:
    return [
        Arm("async_identity", "identity", 1.0, False, False, False, "#4C78A8"),
        Arm("async_topk20", "topk", 0.2, True, False, False, "#F58518"),
        Arm("async_adaptive", "topk", 0.2, True, True, False, "#54A24B"),
        Arm("sync_identity", "identity", 1.0, False, False, True, "#B279A2"),
    ]


def build_config(profile: str) -> ExpConfig:
    config = ExpConfig(profile_name=profile)
    if profile == "quick":
        config.num_workers = 3
        config.target_updates = 60
        config.time_cap_s = 180.0
        config.eval_every = 10
        config.eval_interval_s = 3600.0
        config.eval_batches = 4
        config.batch_size = 4
        config.block_size = 64
        config.n_layer = 2
        config.n_head = 4
        config.n_embd = 96
        config.adaptive_interval_s = 0.75
    elif profile != "report":
        print(f"unknown profile {profile!r}; using report")
        profile = "report"
        config.profile_name = "report"
    overrides = [
        ("ASGC_TARGET", "target_updates", int),
        ("ASGC_BANDWIDTH", "bandwidth_mbps", float),
        ("ASGC_DOWNLINK", "downlink_mbps", float),
        ("ASGC_RHO_MIN", "rho_min", float),
        ("ASGC_RHO_MAX", "rho_max", float),
        ("ASGC_SMAX_MIN", "smax_min", int),
        ("ASGC_SMAX_MAX", "smax_max", int),
        ("ASGC_NUM_WORKERS", "num_workers", int),
    ]
    for env_name, attr, cast in overrides:
        raw = os.environ.get(env_name)
        if raw is not None and raw != "":
            setattr(config, attr, cast(raw))
    config.cuda_strict = os.environ.get("ASGC_CUDA_STRICT", "1") == "1"
    if os.environ.get("ASGC_DISABLE_CUDA") == "1":
        config.cuda_strict = False
    if config.rho_min > config.rho_max:
        raise ValueError("ASGC_RHO_MIN must be <= ASGC_RHO_MAX")
    if config.smax_min > config.smax_max:
        raise ValueError("ASGC_SMAX_MIN must be <= ASGC_SMAX_MAX")
    return config


def parse_seeds(raw: str) -> list[int]:
    seeds = [int(token) for token in raw.split(",") if token.strip()]
    if not seeds:
        raise ValueError("ASGC_SEEDS must contain at least one integer")
    return seeds


def filter_arms(arms: list[Arm], raw: str | None) -> list[Arm]:
    if not raw:
        return arms
    by_name = {arm.name: arm for arm in arms}
    wanted = [token.strip() for token in raw.split(",") if token.strip()]
    unknown = [name for name in wanted if name not in by_name]
    if unknown:
        raise ValueError(f"Unknown ASGC_ARMS entries: {unknown}; choose from {sorted(by_name)}")
    selected = [by_name[name] for name in wanted]
    if not selected:
        raise ValueError("ASGC_ARMS selected no arms")
    return selected


def default_output_dir() -> Path:
    for base in (Path("/content"), Path("/kaggle/working")):
        if base.is_dir():
            return base / "asgc_transformer_results"
    return Path.cwd() / "asgc_transformer_results"


PROFILE = os.environ.get("ASGC_NOTEBOOK_PROFILE", "report").strip().lower()
CONFIG = build_config(PROFILE)
SEEDS = parse_seeds(os.environ.get("ASGC_SEEDS", "0"))
ARMS = filter_arms(make_arms(), os.environ.get("ASGC_ARMS"))
OUTPUT_DIR = Path(os.environ.get("ASGC_OUT", str(default_output_dir()))).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TIME_BUDGET_S = float(os.environ.get("ASGC_TIME_BUDGET", "0") or 0)
AUTO_DOWNLOAD = os.environ.get("ASGC_AUTO_DOWNLOAD", "1") == "1"

print(f"profile: {CONFIG.profile_name}")
print(f"seeds: {SEEDS}")
print(f"arms: {[arm.name for arm in ARMS]}")
print(
    f"model: layers={CONFIG.n_layer} heads={CONFIG.n_head} embd={CONFIG.n_embd} "
    f"block={CONFIG.block_size} batch={CONFIG.batch_size}"
)
print(
    f"run: target_updates={CONFIG.target_updates} workers={CONFIG.num_workers} "
    f"uplink={CONFIG.bandwidth_mbps:g} Mbps downlink={CONFIG.downlink_mbps:g} Mbps "
    f"time_cap={CONFIG.time_cap_s:g}s"
)
print(
    f"adaptive: rho=[{CONFIG.rho_min:g}, {CONFIG.rho_max:g}] "
    f"S_max=[{CONFIG.smax_min}, {CONFIG.smax_max}] interval={CONFIG.adaptive_interval_s:g}s"
)
print("output:", OUTPUT_DIR)
if TIME_BUDGET_S > 0:
    print(f"time budget: {TIME_BUDGET_S:g}s")

## Data and tokenizer

Tiny Shakespeare is downloaded from the canonical char-rnn URL. If the network is unavailable, the notebook generates a synthetic corpus with the same character-level interface. The validation split and validation batches are fixed across seeds so that every seed sees exactly the same evaluation data.

In [ ]:
CORPUS_URLS = [
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt",
]

NOUNS = [
    "river", "castle", "king", "queen", "soldier", "forest", "night", "morning",
    "city", "road", "letter", "storm", "garden", "sword", "crown", "ship",
    "mountain", "village", "fire", "music", "window", "horse", "bridge", "field",
]
VERBS = [
    "saw", "found", "lost", "carried", "remembered", "feared", "welcomed",
    "followed", "wrote", "heard", "left", "guarded", "crossed", "promised",
    "opened", "watched", "returned", "called",
]
ADJS = [
    "old", "quiet", "bright", "dark", "distant", "gentle", "strange", "golden",
    "cold", "hidden", "brave", "lonely", "ancient", "small", "restless",
]
NAMES = ["alice", "martin", "helena", "orsino", "viola", "camillo", "lucius", "julia"]
TEMPLATES = [
    "the {adj} {noun} {verb} the {noun} .\n",
    "{name} {verb} the {adj} {noun} near the {noun} .\n",
    "in the {adj} {noun} , {name} {verb} a {noun} .\n",
    "the {noun} was {adj} and the {noun} was {adj} .\n",
    "{name} said that the {adj} {noun} {verb} the {noun} .\n",
]


def synthetic_corpus(seed: int = 1234, n_sentences: int = 9000) -> str:
    rng = random.Random(seed)
    lines = []
    for _ in range(n_sentences):
        template = rng.choice(TEMPLATES)
        lines.append(
            template.format(
                adj=rng.choice(ADJS),
                noun=rng.choice(NOUNS),
                verb=rng.choice(VERBS),
                name=rng.choice(NAMES),
            )
        )
    return "".join(lines)


def load_corpus(verbose: bool = True) -> tuple[str, str]:
    for url in CORPUS_URLS:
        try:
            with urllib.request.urlopen(url, timeout=20) as response:
                text = response.read().decode("utf-8", errors="ignore")
            if len(text) > 100_000:
                if verbose:
                    print(f"corpus: downloaded {len(text):,} chars from {url}")
                return text[:1_500_000], url
        except Exception as exc:
            if verbose:
                print(f"corpus: {url} unavailable ({type(exc).__name__})")
    text = synthetic_corpus()
    if verbose:
        print(f"corpus: using synthetic fallback ({len(text):,} chars)")
    return text, "synthetic-fallback"


class CharTokenizer:
    def __init__(self, text: str) -> None:
        chars = sorted(set(text))
        self.vocab_size = len(chars)
        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.itos = {i: ch for ch, i in self.stoi.items()}

    def encode(self, text: str) -> torch.Tensor:
        return torch.tensor([self.stoi[ch] for ch in text], dtype=torch.long)

    def decode(self, ids: torch.Tensor) -> str:
        return "".join(self.itos[int(i)] for i in ids.tolist())


def make_windows(ids: torch.Tensor, block_size: int) -> torch.Tensor:
    n = (len(ids) - 1) // block_size
    if n <= 0:
        raise ValueError("corpus too small for the configured block size")
    ids = ids[: n * block_size + 1]
    return ids.unfold(0, block_size + 1, block_size)


class ShardSampler:
    def __init__(self, windows: torch.Tensor, indices: torch.Tensor, batch_size: int, seed: int) -> None:
        self.windows = windows
        self.indices = indices
        self.batch_size = batch_size
        self.generator = torch.Generator().manual_seed(seed)
        self.order: torch.Tensor | None = None
        self.pos = 0

    def _reshuffle(self) -> None:
        perm = torch.randperm(len(self.indices), generator=self.generator)
        self.order = self.indices[perm]
        self.pos = 0

    def next_batch(self) -> tuple[torch.Tensor, torch.Tensor]:
        if self.order is None or self.pos + self.batch_size > len(self.order):
            self._reshuffle()
        idx = self.order[self.pos : self.pos + self.batch_size]
        self.pos += self.batch_size
        seq = self.windows[idx]
        return seq[:, :-1].contiguous(), seq[:, 1:].contiguous()


def make_val_batches(
    windows: torch.Tensor, batch_size: int, max_batches: int, seed: int
) -> list[tuple[torch.Tensor, torch.Tensor]]:
    generator = torch.Generator().manual_seed(seed)
    count = min(len(windows), batch_size * max_batches)
    idx = torch.randperm(len(windows), generator=generator)[:count]
    batches = []
    for start in range(0, count - batch_size + 1, batch_size):
        seq = windows[idx[start : start + batch_size]]
        batches.append((seq[:, :-1].contiguous(), seq[:, 1:].contiguous()))
    return batches


TEXT, CORPUS_SOURCE = load_corpus(verbose=True)
TOKENIZER = CharTokenizer(TEXT)
IDS = TOKENIZER.encode(TEXT)
SPLIT = int(len(IDS) * 0.9)
TRAIN_WINDOWS = make_windows(IDS[:SPLIT], CONFIG.block_size)
VAL_WINDOWS = make_windows(IDS[SPLIT:], CONFIG.block_size)
VAL_BATCHES = make_val_batches(VAL_WINDOWS, CONFIG.batch_size, CONFIG.eval_batches, seed=20240517)

_min_train_windows = CONFIG.batch_size * CONFIG.num_workers * 4
if len(TRAIN_WINDOWS) < _min_train_windows:
    raise RuntimeError(
        f"corpus too small: {len(TRAIN_WINDOWS)} training windows, "
        f"need at least {_min_train_windows}"
    )
if not VAL_BATCHES:
    raise RuntimeError("validation split produced no batches; increase the corpus or batch size")

print(
    f"data: vocab={TOKENIZER.vocab_size} train_windows={len(TRAIN_WINDOWS):,} "
    f"val_windows={len(VAL_WINDOWS):,} val_batches={len(VAL_BATCHES)}"
)

## GPT transformer

The model is a decoder-only, pre-LayerNorm GPT with learned positional embeddings, causal multi-head self-attention, GELU MLPs, and tied input/output embeddings. It uses only `torch.nn`, so the notebook does not depend on `transformers`, `datasets`, or a tokenizer download.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.1
    bias: bool = False


class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        mask = torch.tril(torch.ones(config.block_size, config.block_size))
        self.register_buffer(
            "causal_mask",
            mask.view(1, 1, config.block_size, config.block_size),
            persistent=False,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, channels = x.shape
        q, k, v = self.c_attn(x).split(channels, dim=2)
        q = q.view(batch, seq_len, self.n_head, channels // self.n_head).transpose(1, 2)
        k = k.view(batch, seq_len, self.n_head, channels // self.n_head).transpose(1, 2)
        v = v.view(batch, seq_len, self.n_head, channels // self.n_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.causal_mask[:, :, :seq_len, :seq_len] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = self.attn_dropout(att) @ v
        y = y.transpose(1, 2).contiguous().view(batch, seq_len, channels)
        return self.resid_dropout(self.c_proj(y))


class MLP(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.c_proj(F.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln_1(x))
        return x + self.mlp(self.ln_2(x))


class GPT(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(config.block_size, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.apply(self._init_weights)
        for name, param in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(param, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self, idx: torch.Tensor, targets: torch.Tensor | None = None
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        _, seq_len = idx.shape
        if seq_len > self.config.block_size:
            raise ValueError(
                f"sequence length {seq_len} exceeds block size {self.config.block_size}"
            )
        positions = torch.arange(0, seq_len, dtype=torch.long, device=idx.device)
        x = self.dropout(self.token_embedding(idx) + self.position_embedding(positions))
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    def num_params(self) -> int:
        return sum(p.numel() for p in self.parameters())


def build_template(config: ExpConfig, seed: int) -> GPT:
    torch.manual_seed(seed)
    random.seed(seed)
    gpt_config = GPTConfig(
        vocab_size=TOKENIZER.vocab_size,
        block_size=config.block_size,
        n_layer=config.n_layer,
        n_head=config.n_head,
        n_embd=config.n_embd,
        dropout=config.dropout,
    )
    return GPT(gpt_config)


TEMPLATE = build_template(CONFIG, SEEDS[0])
MODEL_PARAMS = TEMPLATE.num_params()
print(f"model parameters: {MODEL_PARAMS:,}")

## Instrumentation

Two wrappers sit between `DistributedOptimizer` and the in-process transport:

- `RecordingThrottle` injects latency and simulated uplink/downlink bandwidth, and records every push (bytes, status, staleness, service time).
- `CPUOffloadTransport` moves compressed payload tensors to CPU on the worker thread **before** handing them to the CPU parameter-server thread. On CUDA this is what keeps GPU tensors off the server thread.
- `DeviceSafeClient` moves full parameter snapshots onto the worker device (needed when worker 0 owns the GPU).

Wire bytes are read from the library's `payload_bytes` fields, which are computed at encode time, so the measurements do not drift from the actual compressed representation.

In [ ]:
def is_gpu_device(device: Any) -> bool:
    return getattr(device, "type", None) == "cuda"


def maybe_mark_step(device: Any) -> None:
    # CUDA kernels are asynchronous; tensor transfers to CPU synchronize
    # implicitly, so no explicit device barrier is needed here.
    return None


def _to_cpu_piece(payload: Any) -> Any:
    if isinstance(payload, DensePiece):
        return DensePiece(values=payload.values.detach().cpu())
    if isinstance(payload, SparsePiece):
        return SparsePiece(
            indices=payload.indices.detach().cpu(),
            values=payload.values.detach().cpu(),
            total_numel=payload.total_numel,
        )
    if isinstance(payload, QuantizedPiece):
        return QuantizedPiece(
            codes=payload.codes.detach().cpu(),
            scale=payload.scale.detach().cpu(),
            total_numel=payload.total_numel,
        )
    return payload


class CPUOffloadTransport(TransportClient):
    """Move payload tensors to CPU on the worker thread before handoff."""

    def __init__(self, inner: TransportClient) -> None:
        self.inner = inner

    def push(self, update: PushUpdate) -> PushAck:
        buckets = [
            BucketPayload(
                bucket_index=bucket.bucket_index,
                payload=_to_cpu_piece(bucket.payload),
                payload_bytes=bucket.payload_bytes,
                residual_norm=float(bucket.residual_norm),
            )
            for bucket in update.buckets
        ]
        return self.inner.push(
            PushUpdate(update.worker_id, update.version, buckets, update.loss_val)
        )

    def fetch(self, request: FetchRequest) -> FetchResponse:
        return self.inner.fetch(request)

    def close(self) -> None:
        self.inner.close()


class Recorder:
    def __init__(self) -> None:
        self.lock = threading.Lock()
        self.phase = "init"
        self.t0 = time.perf_counter()
        self.push_events: list[dict[str, Any]] = []
        self.init_fetch_bytes = 0
        self.train_fetch_bytes = 0
        self.fetch_count = 0

    def add_push(
        self,
        worker: str,
        num_bytes: int,
        status: str,
        weight: float,
        tau: int,
        service_s: float,
        version: int,
    ) -> None:
        with self.lock:
            self.push_events.append(
                {
                    "phase": self.phase,
                    "t": time.perf_counter() - self.t0,
                    "worker": worker,
                    "bytes": int(num_bytes),
                    "status": status,
                    "weight": float(weight),
                    "tau": int(tau),
                    "service_ms": 1000.0 * float(service_s),
                    "ack_version": int(version),
                }
            )

    def add_fetch(self, num_bytes: int, service_s: float) -> None:
        with self.lock:
            if self.phase == "init":
                self.init_fetch_bytes += int(num_bytes)
            else:
                self.train_fetch_bytes += int(num_bytes)
            self.fetch_count += 1

    def snapshot(self) -> dict[str, Any]:
        with self.lock:
            return {
                "push_events": list(self.push_events),
                "init_fetch_bytes": self.init_fetch_bytes,
                "train_fetch_bytes": self.train_fetch_bytes,
                "fetch_count": self.fetch_count,
            }


class RecordingThrottle(TransportClient):
    def __init__(
        self,
        inner: TransportClient,
        recorder: Recorder,
        worker_id: str,
        uplink_mbps: float,
        downlink_mbps: float,
        latency_s: float,
        stop_event: threading.Event,
    ) -> None:
        self.inner = inner
        self.recorder = recorder
        self.worker_id = worker_id
        self.uplink_mbps = uplink_mbps
        self.downlink_mbps = downlink_mbps
        self.latency_s = latency_s
        self.stop_event = stop_event

    def _sleep(self, seconds: float) -> None:
        if seconds <= 0:
            return
        deadline = time.perf_counter() + seconds
        while True:
            remaining = deadline - time.perf_counter()
            if remaining <= 0 or self.stop_event.is_set():
                return
            time.sleep(min(0.02, remaining))

    def _wire_s(self, num_bytes: int, mbps: float) -> float:
        if mbps <= 0:
            return 0.0
        return num_bytes * 8.0 / (mbps * 1_000_000.0)

    def push(self, update: PushUpdate) -> PushAck:
        self._sleep(self.latency_s)
        self._sleep(self._wire_s(update.total_bytes, self.uplink_mbps))
        started = time.perf_counter()
        ack = self.inner.push(update)
        service_s = time.perf_counter() - started
        accepted = ack.status in ("accepted", "downweighted")
        tau = ack.current_version - (1 if accepted else 0) - update.version
        self.recorder.add_push(
            worker=self.worker_id,
            num_bytes=update.total_bytes,
            status=ack.status,
            weight=ack.weight,
            tau=max(0, int(tau)),
            service_s=service_s,
            version=ack.current_version,
        )
        return ack

    def fetch(self, request: FetchRequest) -> FetchResponse:
        self._sleep(self.latency_s)
        response = self.inner.fetch(request)
        num_bytes = response.encoded_bytes()
        self._sleep(self._wire_s(num_bytes, self.downlink_mbps))
        self.recorder.add_fetch(num_bytes, 0.0)
        return response

    def close(self) -> None:
        self.inner.close()


class DeviceSafeClient(ASGCClient):
    """Move full snapshots onto the worker device (needed for GPU workers)."""

    def sync_model(self, model: nn.Module) -> FetchResponse:
        response = self.fetch_parameters()
        if response.params is not None:
            device = next(model.parameters()).device
            response.params = {
                key: value.to(device=device) for key, value in response.params.items()
            }
        ASGCClient.apply_fetch_response(model, response)
        return response

## CUDA detection and basic preflight

The basic preflight runs one forward/backward/optimizer step on the GPU. This catches device/dtype problems before the full experiment. A second, full-stack preflight (Top-K compression + error feedback + parameter-server push/fetch) runs in the next cell; that one catches CUDA incompatibilities in the compression and bucketing path.

In [ ]:
def detect_cuda() -> tuple[Any, dict[str, Any]]:
    if os.environ.get("ASGC_DISABLE_CUDA") == "1":
        return None, {"available": False, "reason": "ASGC_DISABLE_CUDA=1"}
    if not torch.cuda.is_available():
        return None, {
            "available": False,
            "reason": "torch.cuda.is_available() is False",
            "torch": torch.__version__,
            "torch_cuda": torch.version.cuda,
            "device_count": torch.cuda.device_count(),
        }
    try:
        index = int(os.environ.get("ASGC_CUDA_DEVICE", "0"))
        device = torch.device("cuda", index)
        probe = (torch.ones(4, device=device) + 1).cpu().tolist()
        props = torch.cuda.get_device_properties(index)
        return device, {
            "available": True,
            "device": str(device),
            "device_type": device.type,
            "name": props.name,
            "capability": f"{props.major}.{props.minor}",
            "total_memory_gb": props.total_memory / 1e9,
            "probe": probe,
            "torch": torch.__version__,
            "torch_cuda": torch.version.cuda,
            "device_count": torch.cuda.device_count(),
        }
    except Exception as exc:
        return None, {"available": False, "reason": f"{type(exc).__name__}: {exc}"}


def gpu_preflight(
    template: GPT,
    config: ExpConfig,
    val_batches: list[tuple[torch.Tensor, torch.Tensor]],
    gpu_device: Any,
) -> dict[str, Any]:
    if gpu_device is None:
        return {"ok": False, "skipped": True}
    started = time.perf_counter()
    model = None
    try:
        model = copy.deepcopy(template).to(gpu_device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, foreach=False)
        x, y = val_batches[0]
        x, y = x.to(gpu_device), y.to(gpu_device)
        model.train()
        _, loss = model(x, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        value = float(loss.detach().cpu())
        torch.cuda.synchronize(gpu_device)
        return {
            "ok": True,
            "loss": value,
            "elapsed_s": time.perf_counter() - started,
            "device": str(gpu_device),
            "name": torch.cuda.get_device_name(gpu_device),
        }
    except Exception as exc:
        return {
            "ok": False,
            "error": f"{type(exc).__name__}: {exc}",
            "elapsed_s": time.perf_counter() - started,
        }
    finally:
        if model is not None:
            del model
        gc.collect()


GPU_DEVICE, GPU_INFO = detect_cuda()
print("gpu:", GPU_INFO)
if GPU_DEVICE is None and CONFIG.cuda_strict:
    raise RuntimeError(
        "This notebook requires a CUDA GPU (ASGC_CUDA_STRICT=1 by default), but CUDA is "
        f"unavailable: {GPU_INFO.get('reason')}.\n"
        f"Detected torch {torch.__version__} (CUDA build: {torch.version.cuda}), "
        f"device count: {torch.cuda.device_count()}.\n"
        "1. Select a GPU runtime (Colab: Runtime > Change runtime type > T4/L4/A100; "
        "Kaggle: Accelerator > GPU).\n"
        "2. If the runtime has a GPU but this reports False, the installed PyTorch is a "
        "CPU-only wheel. Install a CUDA build of torch, restart the kernel, and rerun.\n"
        "3. Check `nvidia-smi` and `torch.cuda.is_available()` before running again.\n"
        "To run CPU-only for a smoke test instead, set ASGC_DISABLE_CUDA=1 before "
        "starting the kernel."
    )
GPU_PREFLIGHT: dict[str, Any] = {"ok": False, "skipped": True}
if GPU_DEVICE is not None:
    GPU_PREFLIGHT = gpu_preflight(TEMPLATE, CONFIG, VAL_BATCHES, GPU_DEVICE)
    print("gpu basic preflight:", GPU_PREFLIGHT)
    if not GPU_PREFLIGHT.get("ok"):
        if CONFIG.cuda_strict:
            raise RuntimeError(f"CUDA basic preflight failed: {GPU_PREFLIGHT}")
        print("CUDA basic preflight failed; switching the whole run to CPU.")
        GPU_DEVICE = None
else:
    print("CUDA unavailable; the notebook will run CPU-only.")

## Experiment runner

`run_arm` starts one parameter server thread and `num_workers` worker threads. Each worker:
1. connects over the in-process `LocalTransportHub`,
2. synchronizes the model from the server,
3. runs a warm-up forward/backward to initialize CUDA kernels,
4. enters the timed loop: delay, forward, backward, `DistributedOptimizer.step()`, optional barrier.

The server thread is the only writer to the authoritative model. Validation snapshots are taken under the server lock and evaluated on CPU; the final snapshot is also evaluated on the GPU after all workers have stopped.

The synchronous arm uses the same code path with staleness filtering disabled and a barrier after every push. It is a barrier-synchronous emulation, not a separate optimizer, and it **does not average the round's gradients**: after the barrier each worker's push becomes its own server version. That differs from the research harness's `workload.sync`, which averages one push per worker per round (see findings section 12, item 3).

In [ ]:
def empty_metric_totals() -> dict[str, float]:
    return {"loss_sum": 0.0, "tokens": 0.0, "correct1": 0.0, "correct5": 0.0}


def accumulate_metrics(
    totals: dict[str, float],
    logits: torch.Tensor,
    targets: torch.Tensor,
    loss: torch.Tensor,
) -> None:
    flat_logits = logits.reshape(-1, logits.size(-1))
    flat_targets = targets.reshape(-1)
    n_tokens = int(flat_targets.numel())
    if n_tokens == 0:
        return
    totals["loss_sum"] += float(loss.detach().cpu()) * n_tokens
    totals["tokens"] += n_tokens
    top1 = flat_logits.argmax(dim=-1)
    totals["correct1"] += float(top1.eq(flat_targets).sum().detach().cpu())
    k = min(5, flat_logits.size(-1))
    topk = flat_logits.topk(k, dim=-1).indices
    hits = topk.eq(flat_targets.unsqueeze(-1)).any(dim=-1)
    totals["correct5"] += float(hits.sum().detach().cpu())


def finalize_metrics(totals: dict[str, float]) -> dict[str, float]:
    tokens = max(totals["tokens"], 1.0)
    loss = totals["loss_sum"] / tokens
    return {
        "loss": loss,
        "ppl": math.exp(min(loss, 50.0)),
        "acc1": totals["correct1"] / tokens,
        "acc5": totals["correct5"] / tokens,
        "tokens": int(totals["tokens"]),
    }


@torch.no_grad()
def evaluate_cpu(
    model: GPT, batches: list[tuple[torch.Tensor, torch.Tensor]]
) -> dict[str, float]:
    model.eval()
    totals = empty_metric_totals()
    for x, y in batches:
        logits, loss = model(x, y)
        accumulate_metrics(totals, logits, y, loss)
    model.train()
    return finalize_metrics(totals)


@torch.no_grad()
def evaluate_on_cuda(
    state_dict: dict[str, torch.Tensor],
    template: GPT,
    batches: list[tuple[torch.Tensor, torch.Tensor]],
    device: Any,
) -> dict[str, Any]:
    started = time.perf_counter()
    model = None
    try:
        model = copy.deepcopy(template).to(device)
        moved = {key: value.to(device=device) for key, value in state_dict.items()}
        model.load_state_dict(moved)
        model.eval()
        totals = empty_metric_totals()
        for x, y in batches:
            x, y = x.to(device), y.to(device)
            logits, loss = model(x, y)
            accumulate_metrics(totals, logits, y, loss)
        metrics = finalize_metrics(totals)
        return {
            "ok": True,
            "loss": metrics["loss"],
            "ppl": metrics["ppl"],
            "acc1": metrics["acc1"],
            "acc5": metrics["acc5"],
            "elapsed_s": time.perf_counter() - started,
        }
    except Exception as exc:
        return {
            "ok": False,
            "error": f"{type(exc).__name__}: {exc}",
            "elapsed_s": time.perf_counter() - started,
        }
    finally:
        if model is not None:
            del model
        gc.collect()


def snapshot_cpu(server: ParameterServer, lock: threading.Lock) -> dict[str, torch.Tensor]:
    with lock:
        return {key: value.detach().cpu().clone() for key, value in server.model.state_dict().items()}


def apply_worker_delay(profile: dict[str, float], step: int, rng: random.Random) -> None:
    delay = profile["base"] + rng.random() * profile["jitter"]
    if profile["spike_every"] > 0 and step > 0 and step % int(profile["spike_every"]) == 0:
        delay += profile["spike"]
    if delay > 0:
        time.sleep(delay)


def gpu_stack_preflight(
    template: GPT,
    config: ExpConfig,
    train_windows: torch.Tensor,
    gpu_device: Any,
) -> dict[str, Any]:
    """Exercise Top-K + error feedback + push/fetch on the CUDA worker path."""
    if gpu_device is None:
        return {"ok": False, "skipped": True}
    started = time.perf_counter()
    hub = LocalTransportHub()
    server_model = copy.deepcopy(template)
    server_optimizer = torch.optim.AdamW(server_model.parameters(), lr=config.lr, foreach=False)
    server = ParameterServer(
        model=server_model,
        optimizer=server_optimizer,
        bind_address=hub,
        staleness_config=StalenessConfig(
            enabled=True, s_low=config.s_low, s_max=config.s_max, beta=config.beta
        ),
        compression_mode="topk",
        initial_strength=0.2,
        bucket_cap_mb=config.bucket_cap_mb,
    )
    stop = threading.Event()
    lock = threading.Lock()

    def server_loop() -> None:
        while not stop.is_set():
            with lock:
                worked = server.process_one()
            if not worked:
                time.sleep(0.001)

    server_thread = threading.Thread(target=server_loop, name="gpu-preflight-server", daemon=True)
    server_thread.start()
    client = None
    model = None
    try:
        model = copy.deepcopy(template).to(gpu_device)
        raw = LocalTransportClient(hub, worker_id="gpu_preflight")
        throttle = RecordingThrottle(
            inner=raw,
            recorder=Recorder(),
            worker_id="gpu_preflight",
            uplink_mbps=0.0,
            downlink_mbps=0.0,
            latency_s=0.0,
            stop_event=stop,
        )
        client = DeviceSafeClient(CPUOffloadTransport(throttle), worker_id="gpu_preflight")
        client.sync_model(model)
        base_optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, foreach=False)
        distributed_model = DistributedModel(model, client=client)
        optimizer = DistributedOptimizer(
            base_optimizer=base_optimizer,
            model=distributed_model,
            client=client,
            compression=CompressionConfig(mode="topk", rho=0.2, bits=8),
            error_feedback=True,
            bucket_cap_mb=config.bucket_cap_mb,
            local_steps=1,
            max_grad_norm=config.max_grad_norm,
        )
        batch = train_windows[: config.batch_size]
        x = batch[:, :-1].contiguous().to(gpu_device)
        y = batch[:, 1:].contiguous().to(gpu_device)
        optimizer.zero_grad(set_to_none=True)
        _, loss = model(x, y)
        loss.backward()
        loss_value = float(loss.detach().cpu())
        optimizer.step(loss_val=loss_value)
        optimizer.zero_grad(set_to_none=True)
        maybe_mark_step(gpu_device)
        deadline = time.time() + 15.0
        while server.version < 1 and time.time() < deadline:
            time.sleep(0.02)
        return {
            "ok": server.version >= 1,
            "loss": loss_value,
            "server_version": int(server.version),
            "elapsed_s": time.perf_counter() - started,
        }
    except Exception as exc:
        return {
            "ok": False,
            "error": f"{type(exc).__name__}: {exc}",
            "elapsed_s": time.perf_counter() - started,
        }
    finally:
        if client is not None:
            client.close()
        stop.set()
        server_thread.join(timeout=5)
        server.close()
        del model
        gc.collect()


def run_arm(
    arm: Arm,
    config: ExpConfig,
    template: GPT,
    tokenizer: CharTokenizer,
    train_windows: torch.Tensor,
    val_batches: list[tuple[torch.Tensor, torch.Tensor]],
    gpu_device: Any,
    output_dir: Path,
    verbose: bool = True,
) -> dict[str, Any]:
    del tokenizer
    torch.manual_seed(config.seed + 17)
    random.seed(config.seed + 17)

    server_model = copy.deepcopy(template)
    server_model.train()
    server_optimizer = torch.optim.AdamW(
        server_model.parameters(),
        lr=config.lr,
        betas=(0.9, 0.95),
        weight_decay=config.weight_decay,
        foreach=False,
    )

    adaptive_config = None
    if arm.adaptive:
        adaptive_config = AdaptiveConfig(
            enabled=True,
            interval_s=config.adaptive_interval_s,
            mode="threshold",
            adapt_compression=True,
            adapt_staleness=True,
            rho_bounds=(config.rho_min, config.rho_max),
            max_delta_rho=config.max_delta_rho,
            bits_bounds=(2, 8),
            max_delta_bits=1,
            s_max_bounds=(config.smax_min, config.smax_max),
            max_delta_s_max=config.max_delta_smax,
            worsen_tol=config.worsen_tol,
            variability_tol=config.variability_tol,
            reject_frac_max=config.reject_frac_max,
            comm_pressure_min=config.comm_pressure_min,
            instability_persistence=2,
        )

    hub = LocalTransportHub()
    server = ParameterServer(
        model=server_model,
        optimizer=server_optimizer,
        bind_address=hub,
        staleness_config=StalenessConfig(
            enabled=not arm.sync,
            s_low=config.s_low,
            s_max=config.s_max,
            beta=config.beta,
        ),
        compression_mode=arm.mode,
        initial_strength=arm.rho,
        adaptive_config=adaptive_config,
        bucket_cap_mb=config.bucket_cap_mb,
    )

    stop_signal = threading.Event()
    server_stop = threading.Event()
    server_lock = threading.Lock()

    def server_loop() -> None:
        while not server_stop.is_set():
            with server_lock:
                worked = server.process_one()
            if not worked:
                time.sleep(0.0005)

    server_thread = threading.Thread(target=server_loop, name=f"{arm.name}-server", daemon=True)
    server_thread.start()

    recorder = Recorder()
    ready_barrier = threading.Barrier(config.num_workers + 1)
    start_event = threading.Event()
    run_barrier = threading.Barrier(config.num_workers) if arm.sync else None

    worker_devices: list[Any] = [torch.device("cpu") for _ in range(config.num_workers)]
    if config.use_gpu_worker and gpu_device is not None:
        worker_devices[0] = gpu_device

    worker_errors: list[str] = []
    worker_steps = [0] * config.num_workers
    worker_losses: list[list[float]] = [[] for _ in range(config.num_workers)]
    sampler_generators = [random.Random(config.seed * 100 + i) for i in range(config.num_workers)]

    total_train = len(train_windows)
    base_indices = torch.arange(total_train)
    shard_size = math.ceil(total_train / config.num_workers)
    shards = [
        base_indices[i * shard_size : (i + 1) * shard_size]
        for i in range(config.num_workers)
    ]

    def worker_main(worker_index: int) -> None:
        client = None
        try:
            device = worker_devices[worker_index]
            model = copy.deepcopy(template).to(device)
            worker_id = f"w{worker_index}"
            raw = LocalTransportClient(hub, worker_id=worker_id)
            throttle = RecordingThrottle(
                inner=raw,
                recorder=recorder,
                worker_id=worker_id,
                uplink_mbps=config.bandwidth_mbps,
                downlink_mbps=config.downlink_mbps,
                latency_s=config.latency_s,
                stop_event=stop_signal,
            )
            transport = CPUOffloadTransport(throttle) if is_gpu_device(device) else throttle
            client = DeviceSafeClient(transport, worker_id=worker_id)
            client.sync_model(model)

            base_optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, foreach=False)
            distributed_model = DistributedModel(model, client=client)
            optimizer = DistributedOptimizer(
                base_optimizer=base_optimizer,
                model=distributed_model,
                client=client,
                compression=CompressionConfig(mode=arm.mode, rho=arm.rho, bits=8),
                error_feedback=arm.error_feedback,
                bucket_cap_mb=config.bucket_cap_mb,
                local_steps=config.local_steps,
                max_grad_norm=config.max_grad_norm,
            )
            sampler = ShardSampler(
                train_windows,
                shards[worker_index],
                config.batch_size,
                seed=config.seed * 1000 + worker_index,
            )

            model.train()
            warm_x, warm_y = sampler.next_batch()
            warm_x, warm_y = warm_x.to(device), warm_y.to(device)
            optimizer.zero_grad(set_to_none=True)
            _, warm_loss = model(warm_x, warm_y)
            warm_loss.backward()
            float(warm_loss.detach().cpu())
            maybe_mark_step(device)
            optimizer.zero_grad(set_to_none=True)

            ready_barrier.wait(timeout=600)
            if not start_event.wait(timeout=600):
                return

            loss_ema = None
            max_steps = max(32, int(config.target_updates * 3))
            for step in range(max_steps):
                if stop_signal.is_set() or server.version >= config.target_updates:
                    break
                apply_worker_delay(
                    WORKER_PROFILES[worker_index % len(WORKER_PROFILES)],
                    step,
                    sampler_generators[worker_index],
                )
                x, y = sampler.next_batch()
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad(set_to_none=True)
                _, loss = model(x, y)
                loss.backward()
                loss_value = float(loss.detach().cpu())
                worker_losses[worker_index].append(loss_value)
                loss_ema = loss_value if loss_ema is None else 0.9 * loss_ema + 0.1 * loss_value
                optimizer.step(loss_val=loss_ema)
                maybe_mark_step(device)
                worker_steps[worker_index] += 1
                if run_barrier is not None and not stop_signal.is_set():
                    try:
                        run_barrier.wait(timeout=30)
                    except threading.BrokenBarrierError:
                        break
        except Exception as exc:
            worker_errors.append(f"worker {worker_index}: {type(exc).__name__}: {exc}")
            try:
                ready_barrier.abort()
            except Exception:
                pass
            try:
                if run_barrier is not None:
                    run_barrier.abort()
            except Exception:
                pass
        finally:
            if client is not None:
                client.close()

    workers = [
        threading.Thread(target=worker_main, args=(i,), name=f"{arm.name}-w{i}", daemon=True)
        for i in range(config.num_workers)
    ]
    for thread in workers:
        thread.start()

    setup_ok = True
    try:
        ready_barrier.wait(timeout=600)
    except threading.BrokenBarrierError:
        setup_ok = False

    if not setup_ok or worker_errors:
        stop_signal.set()
        start_event.set()
        for thread in workers:
            thread.join(timeout=10)
        server_stop.set()
        server_thread.join(timeout=5)
        server.close()
        return {
            "arm": arm.name,
            "seed": config.seed,
            "error": "worker setup failed",
            "worker_errors": list(worker_errors),
        }

    eval_model = copy.deepcopy(template)
    eval_model.load_state_dict(template.state_dict())
    init_metrics = evaluate_cpu(eval_model, val_batches)

    recorder.t0 = time.perf_counter()
    recorder.phase = "train"
    run_start = recorder.t0
    server._last_control_time = run_start
    evals: list[list[Any]] = [
        [0.0, 0, init_metrics["loss"], init_metrics["acc1"], init_metrics["acc5"]]
    ]
    monitor_samples: list[dict[str, Any]] = [
        {
            "t": 0.0,
            "version": 0,
            "rho": float(server.current_strength),
            "s_max": int(server.staleness_config.s_max),
        }
    ]
    start_event.set()

    last_eval_version = 0
    last_eval_time = run_start
    monitor_stop = threading.Event()

    def monitor_loop() -> None:
        while not monitor_stop.is_set():
            try:
                monitor_samples.append(
                    {
                        "t": time.perf_counter() - run_start,
                        "version": int(server.version),
                        "rho": float(server.current_strength),
                        "s_max": int(server.staleness_config.s_max),
                    }
                )
            except Exception:
                pass
            time.sleep(0.2)

    monitor_thread = threading.Thread(target=monitor_loop, name=f"{arm.name}-monitor", daemon=True)
    monitor_thread.start()

    while any(thread.is_alive() for thread in workers):
        now = time.perf_counter()
        version = int(server.version)
        elapsed = now - run_start
        if version >= config.target_updates or elapsed >= config.time_cap_s:
            stop_signal.set()
        if version > last_eval_version and (
            version - last_eval_version >= config.eval_every
            or now - last_eval_time >= config.eval_interval_s
        ):
            snapshot = snapshot_cpu(server, server_lock)
            eval_model.load_state_dict(snapshot)
            validation = evaluate_cpu(eval_model, val_batches)
            validation_loss = validation["loss"]
            evals.append(
                [
                    time.perf_counter() - run_start,
                    version,
                    validation_loss,
                    validation["acc1"],
                    validation["acc5"],
                ]
            )
            last_eval_version = version
            last_eval_time = time.perf_counter()
            if verbose:
                print(
                    f"  [{arm.name}] t={evals[-1][0]:6.1f}s v={version:4d} "
                    f"val_loss={validation_loss:.4f} acc1={validation['acc1']:.3f} "
                    f"acc5={validation['acc5']:.3f} rho={server.current_strength:.3f} "
                    f"s_max={server.staleness_config.s_max}"
                )
        time.sleep(0.1)

    wall_s = time.perf_counter() - run_start
    stop_signal.set()
    for thread in workers:
        thread.join(timeout=60)

    drain_deadline = time.perf_counter() + 1.0
    while time.perf_counter() < drain_deadline:
        with server_lock:
            worked = server.process_one()
        if not worked:
            break

    monitor_stop.set()
    monitor_thread.join(timeout=2)
    server_stop.set()
    server_thread.join(timeout=5)
    completed_updates = int(server.version)

    final_snapshot = {
        key: value.detach().cpu().clone()
        for key, value in server.model.state_dict().items()
    }
    eval_model.load_state_dict(final_snapshot)
    final_metrics = evaluate_cpu(eval_model, val_batches)
    final_val_loss = final_metrics["loss"]
    evals.append(
        [
            time.perf_counter() - run_start,
            completed_updates,
            final_val_loss,
            final_metrics["acc1"],
            final_metrics["acc5"],
        ]
    )
    checkpoint_path: str | None = None
    try:
        checkpoint_dir = Path(output_dir) / "checkpoints"
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        checkpoint_file = checkpoint_dir / f"{arm.name}_seed{config.seed}.pt"
        torch.save(
            {
                "arm": arm.name,
                "seed": config.seed,
                "state_dict": final_snapshot,
                "final_val_loss": final_val_loss,
                "final_ppl": final_metrics["ppl"],
                "final_acc1": final_metrics["acc1"],
                "final_acc5": final_metrics["acc5"],
                "config": asdict(config),
            },
            checkpoint_file,
        )
        checkpoint_path = str(checkpoint_file)
    except Exception as exc:
        print(f"  checkpoint save failed: {type(exc).__name__}: {exc}")
    rho_final = float(server.current_strength)
    smax_final = int(server.staleness_config.s_max)
    server.close()

    gpu_eval: dict[str, Any] = {"ok": False, "skipped": True}
    if config.eval_on_gpu and gpu_device is not None:
        gpu_eval = evaluate_on_cuda(final_snapshot, template, val_batches, gpu_device)

    recorder_data = recorder.snapshot()
    pushes = recorder_data["push_events"]
    train_pushes = [event for event in pushes if event["phase"] == "train"]
    status_counts = {"accepted": 0, "downweighted": 0, "rejected": 0}
    for event in train_pushes:
        status_counts[event["status"]] = status_counts.get(event["status"], 0) + 1
    taus = [event["tau"] for event in train_pushes]
    taus_sorted = sorted(taus)
    p95_tau = (
        taus_sorted[min(len(taus_sorted) - 1, int(0.95 * len(taus_sorted)))]
        if taus_sorted
        else 0
    )
    push_bytes = sum(event["bytes"] for event in train_pushes)
    fetch_bytes = recorder_data["train_fetch_bytes"]
    service_ms = [event["service_ms"] for event in train_pushes]

    per_worker: dict[str, dict[str, Any]] = {}
    for index in range(config.num_workers):
        worker_id = f"w{index}"
        events = [event for event in train_pushes if event["worker"] == worker_id]
        per_worker[worker_id] = {
            "device": str(worker_devices[index]),
            "pushes": len(events),
            "accepted": sum(
                1 for event in events if event["status"] in ("accepted", "downweighted")
            ),
            "downweighted": sum(1 for event in events if event["status"] == "downweighted"),
            "rejected": sum(1 for event in events if event["status"] == "rejected"),
            "mean_tau": sum(event["tau"] for event in events) / max(len(events), 1),
            "bytes": sum(event["bytes"] for event in events),
            "mean_service_ms": sum(event["service_ms"] for event in events)
            / max(len(events), 1),
        }

    rho_samples = [sample["rho"] for sample in monitor_samples if sample["version"] > 0]
    smax_samples = [sample["s_max"] for sample in monitor_samples if sample["version"] > 0]

    result = {
        "arm": arm.name,
        "seed": config.seed,
        "mode": arm.mode,
        "sync": arm.sync,
        "error_feedback": arm.error_feedback,
        "adaptive": arm.adaptive,
        "wall_s": wall_s,
        "completed_updates": completed_updates,
        "target_updates": config.target_updates,
        "target_reached": completed_updates >= config.target_updates,
        "push_count": len(train_pushes),
        "status_counts": status_counts,
        "rejection_rate": status_counts.get("rejected", 0) / max(len(train_pushes), 1),
        "downweight_rate": status_counts.get("downweighted", 0) / max(len(train_pushes), 1),
        "mean_tau": sum(taus) / max(len(taus), 1),
        "p95_tau": p95_tau,
        "push_bytes": push_bytes,
        "fetch_bytes_train": fetch_bytes,
        "fetch_bytes_init": recorder_data["init_fetch_bytes"],
        "total_wire_bytes_train": push_bytes + fetch_bytes,
        "bytes_per_push": push_bytes / max(len(train_pushes), 1),
        "uplink_bytes_per_accepted_update": push_bytes / max(completed_updates, 1),
        "downlink_bytes_per_accepted_update": fetch_bytes / max(completed_updates, 1),
        "mean_service_ms": sum(service_ms) / max(len(service_ms), 1),
        "updates_per_s": completed_updates / max(wall_s, 1e-6),
        "worker_steps": list(worker_steps),
        "worker_devices": [str(device) for device in worker_devices],
        "worker_mean_losses": [
            sum(values) / len(values) if values else None for values in worker_losses
        ],
        "per_worker": per_worker,
        "init_val_loss": init_metrics["loss"],
        "init_ppl": init_metrics["ppl"],
        "init_acc1": init_metrics["acc1"],
        "init_acc5": init_metrics["acc5"],
        "final_val_loss": final_metrics["loss"],
        "final_ppl": final_metrics["ppl"],
        "final_acc1": final_metrics["acc1"],
        "final_acc5": final_metrics["acc5"],
        "eval_tokens": final_metrics["tokens"],
        "eval_curve": evals,
        "checkpoint_path": checkpoint_path,
        "rho_initial": float(arm.rho),
        "rho_final": rho_final,
        "rho_mean": (
            sum(rho_samples) / max(len(rho_samples), 1) if rho_samples else float(arm.rho)
        ),
        "smax_final": smax_final,
        "smax_mean": (
            sum(smax_samples) / max(len(smax_samples), 1) if smax_samples else config.s_max
        ),
        "monitor_samples": monitor_samples,
        "gpu_eval": gpu_eval,
        "gpu_worker": is_gpu_device(worker_devices[0]),
        "partial": bool(worker_errors),
        "worker_errors": list(worker_errors),
        "_push_events": pushes,
    }
    return result

## Analysis and aggregation

The default run uses a single seed (`0`), while the analysis remains multi-seed aware: set `ASGC_SEEDS=0,1,2` and each seed gets its own time-to-target threshold, derived from that seed's uncompressed asynchronous final validation loss plus 1%. Cross-arm comparisons are computed per seed and then averaged, so a single lucky or unlucky seed cannot dominate the headline number.

In [ ]:
SUMMARY_KEYS = [
    "wall_s",
    "completed_updates",
    "target_reached",
    "push_count",
    "rejection_rate",
    "downweight_rate",
    "mean_tau",
    "p95_tau",
    "push_bytes",
    "fetch_bytes_train",
    "total_wire_bytes_train",
    "bytes_per_push",
    "uplink_bytes_per_accepted_update",
    "downlink_bytes_per_accepted_update",
    "mean_service_ms",
    "updates_per_s",
    "init_val_loss",
    "init_acc1",
    "init_acc5",
    "final_val_loss",
    "final_ppl",
    "final_acc1",
    "final_acc5",
    "eval_tokens",
    "rho_initial",
    "rho_mean",
    "rho_final",
    "smax_mean",
    "smax_final",
    "time_to_target_s",
    "gpu_worker",
    "partial",
]

AGG_CSV_FIELDS = [
    "wall_s",
    "completed_updates",
    "final_val_loss",
    "final_ppl",
    "final_acc1",
    "final_acc5",
    "push_bytes",
    "fetch_bytes_train",
    "total_wire_bytes_train",
    "uplink_bytes_per_accepted_update",
    "bytes_per_push",
    "mean_tau",
    "p95_tau",
    "rejection_rate",
    "downweight_rate",
    "updates_per_s",
    "rho_mean",
    "rho_final",
    "smax_mean",
    "smax_final",
    "time_to_target_s",
    "gpu_eval_loss",
]


def _ratio(numerator: float, denominator: float) -> float:
    return float(numerator) / max(float(denominator), 1e-9)


def time_to_loss(curve: list[list[Any]], target: float) -> float | None:
    if curve and curve[0][2] <= target:
        return float(curve[0][0])
    for index in range(1, len(curve)):
        point0 = curve[index - 1]
        point1 = curve[index]
        t0, v0 = float(point0[0]), float(point0[2])
        t1, v1 = float(point1[0]), float(point1[2])
        if v1 <= target:
            if v0 <= target:
                return t0
            alpha = (v0 - target) / max(v0 - v1, 1e-9)
            return float(t0 + alpha * (t1 - t0))
    return None


def summarize_seed(results: list[dict[str, Any]], config: ExpConfig) -> dict[str, Any]:
    by_name = {result["arm"]: result for result in results if "error" not in result}
    reference = by_name.get("async_identity")
    if reference is not None:
        target_loss = float(reference["final_val_loss"]) * 1.01
    elif by_name:
        target_loss = min(float(result["final_val_loss"]) for result in by_name.values()) * 1.01
    else:
        target_loss = 0.0

    for result in by_name.values():
        result["time_to_target_s"] = time_to_loss(result["eval_curve"], target_loss)

    arms: dict[str, dict[str, Any]] = {}
    for name, result in by_name.items():
        entry = {key: result.get(key) for key in SUMMARY_KEYS}
        if result.get("gpu_eval", {}).get("ok"):
            entry["gpu_eval_loss"] = float(result["gpu_eval"]["loss"])
        else:
            entry["gpu_eval_loss"] = None
        arms[name] = entry

    comparisons: dict[str, float] = {}
    ident = by_name.get("async_identity")
    topk = by_name.get("async_topk20")
    adapt = by_name.get("async_adaptive")
    sync = by_name.get("sync_identity")

    if ident is not None and sync is not None:
        comparisons["async_vs_sync_speedup"] = _ratio(sync["wall_s"], ident["wall_s"])
        if ident.get("time_to_target_s") and sync.get("time_to_target_s"):
            comparisons["async_time_to_target_speedup"] = _ratio(
                sync["time_to_target_s"], ident["time_to_target_s"]
            )
    if ident is not None and topk is not None:
        comparisons["topk20_uplink_ratio_vs_identity"] = _ratio(
            ident["push_bytes"], topk["push_bytes"]
        )
        comparisons["topk20_total_ratio_vs_identity"] = _ratio(
            ident["total_wire_bytes_train"], topk["total_wire_bytes_train"]
        )
        comparisons["topk20_wall_speedup_vs_identity"] = _ratio(ident["wall_s"], topk["wall_s"])
        comparisons["topk20_delta_final_loss"] = float(topk["final_val_loss"]) - float(
            ident["final_val_loss"]
        )
        comparisons["topk20_delta_final_acc1"] = float(topk["final_acc1"]) - float(
            ident["final_acc1"]
        )
        comparisons["topk20_delta_final_acc5"] = float(topk["final_acc5"]) - float(
            ident["final_acc5"]
        )
    if topk is not None and adapt is not None:
        comparisons["adaptive_uplink_savings_pct_vs_topk20"] = 100.0 * (
            1.0 - _ratio(adapt["push_bytes"], topk["push_bytes"])
        )
        comparisons["adaptive_total_savings_pct_vs_topk20"] = 100.0 * (
            1.0 - _ratio(adapt["total_wire_bytes_train"], topk["total_wire_bytes_train"])
        )
        comparisons["adaptive_uplink_ratio_vs_topk20"] = _ratio(
            topk["push_bytes"], adapt["push_bytes"]
        )
        comparisons["adaptive_wall_speedup_vs_topk20"] = _ratio(topk["wall_s"], adapt["wall_s"])
        comparisons["adaptive_delta_final_loss"] = float(adapt["final_val_loss"]) - float(
            topk["final_val_loss"]
        )
        comparisons["adaptive_delta_final_acc1"] = float(adapt["final_acc1"]) - float(
            topk["final_acc1"]
        )
        comparisons["adaptive_delta_final_acc5"] = float(adapt["final_acc5"]) - float(
            topk["final_acc5"]
        )
        if adapt.get("time_to_target_s") and topk.get("time_to_target_s"):
            comparisons["adaptive_time_to_target_speedup"] = _ratio(
                topk["time_to_target_s"], adapt["time_to_target_s"]
            )
        comparisons["adaptive_rho_reduction"] = float(adapt["rho_initial"]) - float(
            adapt["rho_final"]
        )
        comparisons["adaptive_smax_increase"] = float(adapt["smax_final"]) - float(config.s_max)
    if ident is not None and adapt is not None:
        comparisons["adaptive_uplink_ratio_vs_identity"] = _ratio(
            ident["push_bytes"], adapt["push_bytes"]
        )
        comparisons["adaptive_total_ratio_vs_identity"] = _ratio(
            ident["total_wire_bytes_train"], adapt["total_wire_bytes_train"]
        )
        comparisons["adaptive_wall_speedup_vs_identity"] = _ratio(ident["wall_s"], adapt["wall_s"])
        comparisons["adaptive_delta_final_loss_vs_identity"] = float(
            adapt["final_val_loss"]
        ) - float(ident["final_val_loss"])
        comparisons["adaptive_delta_final_acc1_vs_identity"] = float(
            adapt["final_acc1"]
        ) - float(ident["final_acc1"])
        comparisons["adaptive_delta_final_acc5_vs_identity"] = float(
            adapt["final_acc5"]
        ) - float(ident["final_acc5"])
        if ident.get("time_to_target_s") and adapt.get("time_to_target_s"):
            comparisons["adaptive_time_to_target_speedup_vs_identity"] = _ratio(
                ident["time_to_target_s"], adapt["time_to_target_s"]
            )

    return {
        "seed": config.seed,
        "target_loss_relative": target_loss,
        "arms": arms,
        "comparisons": comparisons,
    }


def aggregate_results(results: list[dict[str, Any]], config: ExpConfig) -> dict[str, Any]:
    completed = [result for result in results if "error" not in result]
    seeds = sorted({int(result["seed"]) for result in completed})
    per_seed: dict[int, dict[str, Any]] = {}
    grouped: dict[str, dict[str, list[float]]] = {}

    for seed in seeds:
        seed_results = [result for result in completed if int(result["seed"]) == seed]
        seed_summary = summarize_seed(seed_results, replace(config, seed=seed))
        per_seed[seed] = seed_summary
        for arm_name, metrics in seed_summary["arms"].items():
            bucket = grouped.setdefault(arm_name, {})
            for key, value in metrics.items():
                if value is None or isinstance(value, bool):
                    continue
                if isinstance(value, (int, float)):
                    bucket.setdefault(key, []).append(float(value))
        for result in seed_results:
            if result.get("gpu_eval", {}).get("ok"):
                grouped.setdefault(result["arm"], {}).setdefault("gpu_eval_loss", []).append(
                    float(result["gpu_eval"]["loss"])
                )

    aggregate_arms: dict[str, dict[str, dict[str, Any]]] = {}
    for arm_name, metrics in grouped.items():
        aggregate_arms[arm_name] = {}
        for key, values in metrics.items():
            aggregate_arms[arm_name][key] = {
                "mean": statistics.fmean(values) if values else None,
                "std": statistics.stdev(values) if len(values) > 1 else 0.0,
                "n": len(values),
                "values": values,
            }
        seeds_with_arm = len(
            {
                int(result["seed"])
                for result in completed
                if result["arm"] == arm_name
            }
        )
        aggregate_arms[arm_name]["n_seeds"] = {
            "mean": float(seeds_with_arm),
            "std": 0.0,
            "n": seeds_with_arm,
            "values": [float(seeds_with_arm)],
        }

    comparison_values: dict[str, list[float]] = {}
    for seed_summary in per_seed.values():
        for key, value in seed_summary["comparisons"].items():
            if isinstance(value, (int, float)) and not isinstance(value, bool):
                comparison_values.setdefault(key, []).append(float(value))
    comparisons = {
        key: {
            "mean": statistics.fmean(values),
            "std": statistics.stdev(values) if len(values) > 1 else 0.0,
            "n": len(values),
            "values": values,
        }
        for key, values in comparison_values.items()
    }

    target_losses = [summary["target_loss_relative"] for summary in per_seed.values()]
    return {
        "profile": config.profile_name,
        "seeds": seeds,
        "target_loss_relative": statistics.fmean(target_losses) if target_losses else 0.0,
        "target_loss_relative_std": (
            statistics.stdev(target_losses) if len(target_losses) > 1 else 0.0
        ),
        "arms": aggregate_arms,
        "comparisons": comparisons,
        "per_seed": {str(seed): summary for seed, summary in per_seed.items()},
        "config": asdict(config),
        "n_results": len(completed),
        "n_errors": sum(1 for result in results if "error" in result),
    }


def _fmt_metric(
    summary: dict[str, Any],
    arm_name: str,
    key: str,
    digits: int = 2,
    scale: float = 1.0,
    suffix: str = "",
) -> str:
    entry = summary["arms"].get(arm_name, {}).get(key)
    if entry is None or entry.get("mean") is None:
        return "n/a"
    return f"{entry['mean'] * scale:.{digits}f} +/- {entry['std'] * scale:.{digits}f}{suffix}"


def format_table(summary: dict[str, Any]) -> str:
    arm_order = [arm.name for arm in ARMS if arm.name in summary["arms"]]
    arm_order += [name for name in summary["arms"] if name not in arm_order]
    if not arm_order:
        return "No completed arms."
    lines = [
        "| arm | wall (s) | updates | val loss | ppl | acc1 % | acc5 % | uplink MB | downlink MB | total MB | mean tau | reject % | rho mean | S_max mean | time-to-target (s) |",
        "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|",
    ]
    for arm_name in arm_order:
        partial = summary["arms"][arm_name].get("partial", {}).get("mean", 0.0) > 0.0
        label = arm_name + (" *" if partial else "")
        lines.append(
            "| "
            + " | ".join(
                [
                    label,
                    _fmt_metric(summary, arm_name, "wall_s", 1),
                    _fmt_metric(summary, arm_name, "completed_updates", 0),
                    _fmt_metric(summary, arm_name, "final_val_loss", 4),
                    _fmt_metric(summary, arm_name, "final_ppl", 2),
                    _fmt_metric(summary, arm_name, "final_acc1", 2, 100.0),
                    _fmt_metric(summary, arm_name, "final_acc5", 2, 100.0),
                    _fmt_metric(summary, arm_name, "push_bytes", 2, 1e-6),
                    _fmt_metric(summary, arm_name, "fetch_bytes_train", 2, 1e-6),
                    _fmt_metric(summary, arm_name, "total_wire_bytes_train", 2, 1e-6),
                    _fmt_metric(summary, arm_name, "mean_tau", 2),
                    _fmt_metric(summary, arm_name, "rejection_rate", 1, 100.0),
                    _fmt_metric(summary, arm_name, "rho_mean", 3),
                    _fmt_metric(summary, arm_name, "smax_mean", 2),
                    _fmt_metric(summary, arm_name, "time_to_target_s", 1),
                ]
            )
            + " |"
        )
    lines.append("")
    lines.append("`*` marks an arm with at least one partial seed (a worker failed but the remaining workers produced metrics).")
    return "\n".join(lines)


def _clean_result(result: dict[str, Any]) -> dict[str, Any]:
    return {key: value for key, value in result.items() if not key.startswith("_")}


def save_outputs(
    results: list[dict[str, Any]],
    summary: dict[str, Any],
    config: ExpConfig,
    environment: dict[str, Any],
    output_dir: Path,
) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)

    with (output_dir / "results.json").open("w", encoding="utf-8") as handle:
        json.dump(
            {
                "summary": summary,
                "results": [_clean_result(result) for result in results],
                "environment": environment,
            },
            handle,
            indent=2,
        )

    per_seed_fields = [
        "seed",
        "arm",
        "wall_s",
        "completed_updates",
        "target_reached",
        "push_count",
        "rejection_rate",
        "downweight_rate",
        "mean_tau",
        "p95_tau",
        "push_bytes",
        "fetch_bytes_train",
        "total_wire_bytes_train",
        "bytes_per_push",
        "uplink_bytes_per_accepted_update",
        "downlink_bytes_per_accepted_update",
        "updates_per_s",
        "init_val_loss",
        "init_acc1",
        "init_acc5",
        "final_val_loss",
        "final_ppl",
        "final_acc1",
        "final_acc5",
        "eval_tokens",
        "rho_mean",
        "rho_final",
        "smax_mean",
        "smax_final",
        "time_to_target_s",
        "partial",
        "gpu_worker",
        "gpu_eval_loss",
    ]
    with (output_dir / "summary_per_seed.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=per_seed_fields)
        writer.writeheader()
        for result in results:
            if "error" in result:
                continue
            row = {key: result.get(key) for key in per_seed_fields}
            row["seed"] = result.get("seed")
            row["arm"] = result.get("arm")
            row["gpu_eval_loss"] = (
                result.get("gpu_eval", {}).get("loss")
                if result.get("gpu_eval", {}).get("ok")
                else None
            )
            writer.writerow(row)

    aggregate_header = ["arm", "n_seeds"]
    for field in AGG_CSV_FIELDS:
        aggregate_header += [f"{field}_mean", f"{field}_std"]
    with (output_dir / "summary_aggregate.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=aggregate_header)
        writer.writeheader()
        for arm_name, metrics in summary["arms"].items():
            row: dict[str, Any] = {
                "arm": arm_name,
                "n_seeds": metrics.get("n_seeds", {}).get("n", 0),
            }
            for field in AGG_CSV_FIELDS:
                entry = metrics.get(field)
                row[f"{field}_mean"] = entry.get("mean") if entry else None
                row[f"{field}_std"] = entry.get("std") if entry else None
            writer.writerow(row)

    with (output_dir / "eval_curves.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["arm", "seed", "t_s", "version", "val_loss", "acc1", "acc5"])
        for result in results:
            if "error" in result:
                continue
            for point in result.get("eval_curve", []):
                if len(point) >= 5:
                    t, version, loss, acc1, acc5 = point[:5]
                else:
                    t, version, loss = point[:3]
                    acc1 = acc5 = None
                writer.writerow([result["arm"], result["seed"], t, version, loss, acc1, acc5])

    with (output_dir / "push_events.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=[
                "arm",
                "seed",
                "phase",
                "t",
                "worker",
                "bytes",
                "status",
                "weight",
                "tau",
                "service_ms",
                "ack_version",
            ],
        )
        writer.writeheader()
        for result in results:
            if "error" in result:
                continue
            for event in result.get("_push_events", []):
                row = dict(event)
                row["arm"] = result["arm"]
                row["seed"] = result["seed"]
                writer.writerow(row)

    with (output_dir / "monitor_samples.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["arm", "seed", "t", "version", "rho", "s_max"])
        writer.writeheader()
        for result in results:
            if "error" in result:
                continue
            for sample in result.get("monitor_samples", []):
                row = dict(sample)
                row["arm"] = result["arm"]
                row["seed"] = result["seed"]
                writer.writerow(row)

    with (output_dir / "worker_summary.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=[
                "arm",
                "seed",
                "worker",
                "device",
                "pushes",
                "accepted",
                "downweighted",
                "rejected",
                "mean_tau",
                "bytes",
                "mean_service_ms",
            ],
        )
        writer.writeheader()
        for result in results:
            if "error" in result:
                continue
            for worker_id, metrics in result.get("per_worker", {}).items():
                row = dict(metrics)
                row["arm"] = result["arm"]
                row["seed"] = result["seed"]
                row["worker"] = worker_id
                writer.writerow(row)

    with (output_dir / "config.json").open("w", encoding="utf-8") as handle:
        json.dump(
            {
                "profile": config.profile_name,
                "config": asdict(config),
                "arms": [asdict(arm) for arm in ARMS],
                "seeds": list(SEEDS),
                "output_dir": str(output_dir),
            },
            handle,
            indent=2,
        )
    with (output_dir / "environment.json").open("w", encoding="utf-8") as handle:
        json.dump(environment, handle, indent=2)
    return output_dir

In [ ]:
def _curve_arrays(curve: list[list[Any]], value_index: int = 2) -> tuple[Any, Any]:
    import numpy as np

    points = sorted(curve, key=lambda item: item[1])
    xs: list[float] = []
    ys: list[float] = []
    for point in points:
        version = float(point[1])
        value = point[value_index] if value_index < len(point) else None
        if value is None:
            continue
        if xs and version <= xs[-1]:
            ys[-1] = float(value)
        else:
            xs.append(version)
            ys.append(float(value))
    return np.asarray(xs, dtype=float), np.asarray(ys, dtype=float)


def _interp_curve(curve: list[list[Any]], grid: Any, value_index: int = 2) -> Any:
    import numpy as np

    xs, ys = _curve_arrays(curve, value_index)
    if len(xs) == 0:
        return np.full_like(grid, np.nan, dtype=float)
    return np.interp(grid, xs, ys, left=np.nan, right=np.nan)


def _series_arrays(samples: list[dict[str, Any]], key: str) -> tuple[Any, Any]:
    import numpy as np

    points = sorted(samples, key=lambda item: item["t"])
    xs = np.asarray([float(item["t"]) for item in points], dtype=float)
    ys = np.asarray([float(item[key]) for item in points], dtype=float)
    return xs, ys


def make_plots(
    results: list[dict[str, Any]],
    summary: dict[str, Any],
    config: ExpConfig,
    output_dir: Path,
) -> list[Path]:
    try:
        import numpy as np
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except Exception as exc:
        print(f"plots skipped: {exc}")
        return []

    warnings.filterwarnings("ignore", category=RuntimeWarning)
    completed = [result for result in results if "error" not in result]
    if not completed:
        print("plots skipped: no completed results")
        return []

    plot_dir = output_dir / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)
    arm_order = [arm.name for arm in ARMS if arm.name in summary["arms"]]
    arm_order += [name for name in summary["arms"] if name not in arm_order]
    colors = {
        arm.name: arm.color for arm in make_arms()
    }
    paths: list[Path] = []

    grid = np.linspace(0, max(1, config.target_updates), 80)
    fig, ax = plt.subplots(figsize=(9, 4.8))
    for arm_name in arm_order:
        curves = []
        for seed in summary["seeds"]:
            matches = [
                result
                for result in completed
                if result["arm"] == arm_name and int(result["seed"]) == int(seed)
            ]
            if matches:
                curves.append(_interp_curve(matches[0]["eval_curve"], grid))
        if not curves:
            continue
        matrix = np.vstack(curves)
        mean = np.nanmean(matrix, axis=0)
        std = np.nanstd(matrix, axis=0)
        color = colors.get(arm_name, "#888888")
        ax.plot(grid, mean, label=arm_name, color=color, linewidth=2)
        if len(curves) == 1:
            matches = [
                result
                for result in completed
                if result["arm"] == arm_name and int(result["seed"]) == int(summary["seeds"][0])
            ]
            if matches:
                xs_raw, ys_raw = _curve_arrays(matches[0]["eval_curve"])
                ax.plot(xs_raw, ys_raw, "o", color=color, markersize=3, alpha=0.8)
        if len(curves) > 1:
            ax.fill_between(grid, mean - std, mean + std, color=color, alpha=0.15)
    target = summary.get("target_loss_relative")
    if target:
        ax.axhline(target, color="black", linestyle="--", linewidth=1, label="target loss")
    ax.set_xlabel("accepted server updates")
    ax.set_ylabel("validation loss")
    seed_count = len(summary["seeds"])
    ax.set_title(
        f"Validation loss vs accepted updates ({seed_count} seed{'s' if seed_count != 1 else ''})"
    )
    ax.legend(fontsize=8)
    fig.tight_layout()
    path = plot_dir / "loss_vs_updates.png"
    fig.savefig(path, dpi=160)
    plt.close(fig)
    paths.append(path)

    fig, ax = plt.subplots(figsize=(9, 4.8))
    for arm_name in arm_order:
        acc1_curves = []
        acc5_curves = []
        for seed in summary["seeds"]:
            matches = [
                result
                for result in completed
                if result["arm"] == arm_name and int(result["seed"]) == int(seed)
            ]
            if matches:
                acc1_curves.append(_interp_curve(matches[0]["eval_curve"], grid, 3))
                acc5_curves.append(_interp_curve(matches[0]["eval_curve"], grid, 4))
        if not acc1_curves:
            continue
        color = colors.get(arm_name, "#888888")
        acc1_mean = np.nanmean(np.vstack(acc1_curves), axis=0)
        acc5_mean = np.nanmean(np.vstack(acc5_curves), axis=0)
        ax.plot(grid, 100.0 * acc1_mean, label=f"{arm_name} top-1", color=color, linewidth=2)
        ax.plot(
            grid,
            100.0 * acc5_mean,
            label=f"{arm_name} top-5",
            color=color,
            linewidth=1.2,
            linestyle="--",
            alpha=0.8,
        )
    ax.set_xlabel("accepted server updates")
    ax.set_ylabel("next-token accuracy (%)")
    ax.set_title("Top-1 and top-5 next-token accuracy vs accepted updates")
    ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
    path = plot_dir / "accuracy_vs_updates.png"
    fig.savefig(path, dpi=160)
    plt.close(fig)
    paths.append(path)

    names = [name for name in arm_order if summary["arms"][name].get("final_acc1")]
    if names:
        fig, ax = plt.subplots(figsize=(9, 4.6))
        x = np.arange(len(names), dtype=float)
        width = 0.38
        acc1 = [100.0 * summary["arms"][name]["final_acc1"]["mean"] for name in names]
        acc5 = [100.0 * summary["arms"][name]["final_acc5"]["mean"] for name in names]
        acc1_std = [100.0 * summary["arms"][name]["final_acc1"]["std"] for name in names]
        acc5_std = [100.0 * summary["arms"][name]["final_acc5"]["std"] for name in names]
        bars1 = ax.bar(
            x - width / 2, acc1, width, yerr=acc1_std, capsize=3, label="top-1", color="#4C72B0"
        )
        bars5 = ax.bar(
            x + width / 2,
            acc5,
            width,
            yerr=acc5_std,
            capsize=3,
            label="top-5",
            color="#DD8452",
            alpha=0.85,
        )
        for bar, value in zip(bars1, acc1):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.6,
                f"{value:.2f}",
                ha="center",
                va="bottom",
                fontsize=7,
            )
        for bar, value in zip(bars5, acc5):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.6,
                f"{value:.2f}",
                ha="center",
                va="bottom",
                fontsize=7,
            )
        ax.set_xticks(x)
        ax.set_xticklabels(names, rotation=20)
        ax.set_ylabel("final next-token accuracy (%)")
        ax.set_title("Final accuracy by arm at the accepted-update budget")
        ax.legend(fontsize=8)
        fig.tight_layout()
        path = plot_dir / "final_accuracy_bar.png"
        fig.savefig(path, dpi=160)
        plt.close(fig)
        paths.append(path)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    names = [name for name in arm_order if summary["arms"][name].get("wall_s")]
    means = [summary["arms"][name]["wall_s"]["mean"] for name in names]
    stds = [summary["arms"][name]["wall_s"]["std"] for name in names]
    ax.bar(names, means, yerr=stds, capsize=3, color=[colors.get(name, "#888888") for name in names])
    ax.set_ylabel("wall-clock seconds")
    ax.set_title(f"Wall-clock to {config.target_updates} accepted updates")
    ax.tick_params(axis="x", rotation=20)
    for index, value in enumerate(means):
        ax.text(index, value, f"{value:.1f}s", ha="center", va="bottom", fontsize=8)
    fig.tight_layout()
    path = plot_dir / "wall_clock.png"
    fig.savefig(path, dpi=160)
    plt.close(fig)
    paths.append(path)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    names = [name for name in arm_order if summary["arms"][name].get("push_bytes")]
    uplink = [summary["arms"][name]["push_bytes"]["mean"] / 1e6 for name in names]
    downlink = [summary["arms"][name]["fetch_bytes_train"]["mean"] / 1e6 for name in names]
    total = [u + d for u, d in zip(uplink, downlink)]
    total_std = [
        float(
            np.hypot(
                summary["arms"][name]["push_bytes"]["std"],
                summary["arms"][name]["fetch_bytes_train"]["std"],
            )
            / 1e6
        )
        for name in names
    ]
    ax.bar(names, uplink, label="uplink (gradients)", color=[colors.get(name, "#888888") for name in names])
    ax.bar(names, downlink, bottom=uplink, label="downlink (deltas)", color="#BBBBBB", alpha=0.8)
    ax.errorbar(names, total, yerr=total_std, fmt="none", ecolor="black", capsize=3)
    ax.set_ylabel("megabytes over the run")
    ax.set_title("Wire bytes at equal accepted-update budget")
    ax.tick_params(axis="x", rotation=20)
    ax.legend()
    fig.tight_layout()
    path = plot_dir / "wire_bytes.png"
    fig.savefig(path, dpi=160)
    plt.close(fig)
    paths.append(path)

    names = [
        name
        for name in arm_order
        if summary["arms"][name].get("time_to_target_s", {}).get("mean") is not None
    ]
    if names:
        fig, ax = plt.subplots(figsize=(9, 4.5))
        means = [summary["arms"][name]["time_to_target_s"]["mean"] for name in names]
        stds = [summary["arms"][name]["time_to_target_s"]["std"] for name in names]
        ax.bar(names, means, yerr=stds, capsize=3, color=[colors.get(name, "#888888") for name in names])
        ax.set_ylabel("seconds to target loss")
        ax.set_title("Time to the uncompressed-async target loss (mean +/- std)")
        ax.tick_params(axis="x", rotation=20)
        fig.tight_layout()
        path = plot_dir / "time_to_target.png"
        fig.savefig(path, dpi=160)
        plt.close(fig)
        paths.append(path)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    names = [name for name in arm_order if summary["arms"][name].get("rejection_rate")]
    rates = [100.0 * summary["arms"][name]["rejection_rate"]["mean"] for name in names]
    ax.bar(names, rates, color=[colors.get(name, "#888888") for name in names])
    ax.set_ylabel("rejected updates (%)")
    ax.set_title("Staleness rejections")
    ax.tick_params(axis="x", rotation=20)
    fig.tight_layout()
    path = plot_dir / "rejection_rate.png"
    fig.savefig(path, dpi=160)
    plt.close(fig)
    paths.append(path)

    adaptive_results = [result for result in completed if result.get("adaptive") and result.get("monitor_samples")]
    if adaptive_results:
        max_t = max(float(result["monitor_samples"][-1]["t"]) for result in adaptive_results)
        time_grid = np.linspace(0, max(0.1, max_t), 100)
        rho_curves = []
        smax_curves = []
        for result in adaptive_results:
            xs, rho = _series_arrays(result["monitor_samples"], "rho")
            _, smax = _series_arrays(result["monitor_samples"], "s_max")
            rho_curves.append(np.interp(time_grid, xs, rho, left=rho[0], right=rho[-1]))
            smax_curves.append(np.interp(time_grid, xs, smax, left=smax[0], right=smax[-1]))
        rho_matrix = np.vstack(rho_curves)
        smax_matrix = np.vstack(smax_curves)
        fig, ax1 = plt.subplots(figsize=(9, 4.6))
        rho_mean = np.mean(rho_matrix, axis=0)
        rho_std = np.std(rho_matrix, axis=0)
        ax1.plot(time_grid, rho_mean, color="#54A24B", label="rho (active fraction)")
        if len(rho_curves) > 1:
            ax1.fill_between(time_grid, rho_mean - rho_std, rho_mean + rho_std, color="#54A24B", alpha=0.15)
        ax1.set_xlabel("wall-clock seconds")
        ax1.set_ylabel("Top-K active fraction rho", color="#54A24B")
        ax1.tick_params(axis="y", labelcolor="#54A24B")
        ax2 = ax1.twinx()
        smax_mean = np.mean(smax_matrix, axis=0)
        smax_std = np.std(smax_matrix, axis=0)
        ax2.plot(time_grid, smax_mean, color="#E45756", label="S_max")
        if len(smax_curves) > 1:
            ax2.fill_between(time_grid, smax_mean - smax_std, smax_mean + smax_std, color="#E45756", alpha=0.15)
        ax2.set_ylabel("staleness bound S_max", color="#E45756")
        ax2.tick_params(axis="y", labelcolor="#E45756")
        ax1.set_title("Adaptive controller trajectory (mean +/- std across seeds)")
        fig.tight_layout()
        path = plot_dir / "adaptive_trajectory.png"
        fig.savefig(path, dpi=160)
        plt.close(fig)
        paths.append(path)

    fig, ax = plt.subplots(figsize=(9, 4.8))
    any_points = False
    for arm_name in arm_order:
        x_entry = summary["arms"][arm_name].get("uplink_bytes_per_accepted_update")
        y_entry = summary["arms"][arm_name].get("final_val_loss")
        if not x_entry or not y_entry:
            continue
        any_points = True
        color = colors.get(arm_name, "#888888")
        ax.errorbar(
            x_entry["mean"] / 1e6,
            y_entry["mean"],
            xerr=x_entry["std"] / 1e6,
            yerr=y_entry["std"],
            fmt="o",
            markersize=7,
            capsize=3,
            color=color,
            label=arm_name,
        )
        ax.annotate(arm_name, (x_entry["mean"] / 1e6, y_entry["mean"]), textcoords="offset points", xytext=(6, 5), fontsize=8)
    if any_points:
        ax.set_xlabel("uplink MB per accepted update")
        ax.set_ylabel("final validation loss")
        ax.set_title("Bandwidth-quality trade-off (mean +/- std across seeds)")
        ax.legend(fontsize=8)
        fig.tight_layout()
        path = plot_dir / "bytes_vs_loss.png"
        fig.savefig(path, dpi=160)
        paths.append(path)
    plt.close(fig)
    return paths

In [ ]:
def _comp(summary: dict[str, Any], key: str, digits: int = 2, unit: str = "") -> str:
    entry = summary["comparisons"].get(key)
    if not entry or entry.get("mean") is None:
        return "not available"
    return (
        f"{entry['mean']:.{digits}f}{unit} "
        f"(std {entry['std']:.{digits}f}, n={entry['n']})"
    )


def _pp(summary: dict[str, Any], key: str, digits: int = 2) -> str:
    entry = summary["comparisons"].get(key)
    if not entry or entry.get("mean") is None:
        return "not available"
    return (
        f"{100.0 * entry['mean']:.{digits}f} pp "
        f"(std {100.0 * entry['std']:.{digits}f}, n={entry['n']})"
    )


def _arm_mean(summary: dict[str, Any], arm_name: str, key: str) -> float | None:
    entry = summary["arms"].get(arm_name, {}).get(key)
    return None if entry is None else entry.get("mean")


def _fmt_value(value: float | None, digits: int = 3) -> str:
    return "n/a" if value is None else f"{value:.{digits}f}"


def _pct(summary: dict[str, Any], arm_name: str, key: str, digits: int = 2) -> str:
    value = _arm_mean(summary, arm_name, key)
    return "n/a" if value is None else f"{100.0 * value:.{digits}f}"


def findings_text(
    results: list[dict[str, Any]],
    summary: dict[str, Any],
    environment: dict[str, Any],
) -> str:
    if not summary.get("arms"):
        return "# Transformer ASGC Experiment - Findings\n\nNo completed results were produced. Check the worker errors in `results.json`.\n"

    config = summary.get("config", {})
    gpu = environment.get("gpu", {}) or {}
    basic = environment.get("gpu_preflight", {}) or {}
    stack = environment.get("gpu_stack_preflight", {}) or {}
    corpus_chars = int(environment.get("corpus_chars") or 0)
    model_params = int(environment.get("model_params") or 0)

    lines = [
        "# Transformer ASGC Experiment - Findings",
        "",
        f"- Runtime: {environment.get('platform')}; Python {environment.get('python')}; torch {environment.get('torch')}",
        f"- GPU: {gpu.get('device', 'unavailable')} ({gpu.get('name', 'unknown')}); "
        f"CUDA {gpu.get('torch_cuda')}; basic preflight={basic.get('ok')}; stack preflight={stack.get('ok')}",
        f"- GPU worker used: {environment.get('gpu_worker_used')}; final GPU evaluation: {environment.get('config', {}).get('eval_on_gpu')}",
        f"- Corpus: {environment.get('corpus_source')} ({corpus_chars:,} chars)",
        f"- Model: {model_params:,} parameters; block={config.get('block_size')}; layers={config.get('n_layer')}; "
        f"heads={config.get('n_head')}; embd={config.get('n_embd')}",
        f"- Run: seeds={summary.get('seeds')}; workers={config.get('num_workers')}; "
        f"target_updates={config.get('target_updates')}; uplink={config.get('bandwidth_mbps')} Mbps; "
        f"downlink={config.get('downlink_mbps')} Mbps",
        f"- Repo commit: {environment.get('repo_commit')}",
        "",
        "## Arm results",
        "",
        format_table(summary),
        "",
        "## Headline comparisons",
        "",
        f"- Asynchronous vs barrier-synchronous (uncompressed): **{_comp(summary, 'async_vs_sync_speedup', unit='x')}** wall-clock speedup.",
        f"- Top-K 20% + EF vs uncompressed async: **{_comp(summary, 'topk20_wall_speedup_vs_identity', unit='x')}** wall-clock, "
        f"**{_comp(summary, 'topk20_uplink_ratio_vs_identity', unit='x')}** uplink reduction, "
        f"delta validation loss **{_comp(summary, 'topk20_delta_final_loss', 4)}**, "
        f"delta top-1 accuracy **{_pp(summary, 'topk20_delta_final_acc1')}**.",
        f"- Final top-1 / top-5 next-token accuracy by arm: "
        f"`async_identity` {_pct(summary, 'async_identity', 'final_acc1')}% / "
        f"{_pct(summary, 'async_identity', 'final_acc5')}%, "
        f"`async_topk20` {_pct(summary, 'async_topk20', 'final_acc1')}% / "
        f"{_pct(summary, 'async_topk20', 'final_acc5')}%, "
        f"`async_adaptive` {_pct(summary, 'async_adaptive', 'final_acc1')}% / "
        f"{_pct(summary, 'async_adaptive', 'final_acc5')}%, "
        f"`sync_identity` {_pct(summary, 'sync_identity', 'final_acc1')}% / "
        f"{_pct(summary, 'sync_identity', 'final_acc5')}%. "
        "The random baseline is about 1.54% top-1 and 7.69% top-5 for a 65-character vocabulary.",
        f"- Adaptive vs fixed Top-K 20% + EF: **{_comp(summary, 'adaptive_uplink_savings_pct_vs_topk20', 1, '%')}** fewer uplink bytes, "
        f"**{_comp(summary, 'adaptive_total_savings_pct_vs_topk20', 1, '%')}** fewer total wire bytes, "
        f"**{_comp(summary, 'adaptive_wall_speedup_vs_topk20', unit='x')}** wall-clock, "
        f"delta validation loss **{_comp(summary, 'adaptive_delta_final_loss', 4)}**, "
        f"delta top-1 accuracy **{_pp(summary, 'adaptive_delta_final_acc1')}**.",
        f"- Adaptive time-to-target vs uncompressed async (target = 1% above uncompressed final loss): **{_comp(summary, 'adaptive_time_to_target_speedup_vs_identity', unit='x')}**.",
    ]
    if summary["comparisons"].get("adaptive_time_to_target_speedup", {}).get("mean") is not None:
        lines.append(
            f"- Adaptive time-to-target vs fixed Top-K: **{_comp(summary, 'adaptive_time_to_target_speedup', unit='x')}**."
        )
    else:
        lines.append(
            "- Adaptive time-to-target vs fixed Top-K: **not available** (Top-K did not reach the uncompressed target within the run)."
        )
    lines += [
        f"- Adaptive vs uncompressed async: **{_comp(summary, 'adaptive_wall_speedup_vs_identity', unit='x')}** wall-clock, "
        f"**{_comp(summary, 'adaptive_total_ratio_vs_identity', unit='x')}** total wire reduction.",
        "",
        "## Adaptive controller behavior",
        "",
    ]

    adaptive = summary["arms"].get("async_adaptive")
    if adaptive:
        lines += [
            f"- Mean `rho` over the run: **{_arm_mean(summary, 'async_adaptive', 'rho_mean'):.3f}**; "
            f"final `rho`: **{_arm_mean(summary, 'async_adaptive', 'rho_final'):.3f}** (started at 0.200).",
            f"- Mean `S_max`: **{_arm_mean(summary, 'async_adaptive', 'smax_mean'):.2f}**; "
            f"final `S_max`: **{_arm_mean(summary, 'async_adaptive', 'smax_final'):.2f}** (started at {config.get('s_max')}).",
            f"- Mean rejection rate: **{100.0 * (_arm_mean(summary, 'async_adaptive', 'rejection_rate') or 0.0):.2f}%**; "
            f"mean staleness: **{_arm_mean(summary, 'async_adaptive', 'mean_tau'):.2f}**.",
        ]
        delta = summary["comparisons"].get("adaptive_delta_final_loss")
        if delta:
            d = delta["mean"]
            if d <= -0.01:
                verdict = f"improved final validation loss by {-d:.4f}"
            elif d <= 0.01:
                verdict = f"kept final validation loss within noise (delta {d:+.4f})"
            elif d <= 0.05:
                verdict = f"cost a small amount of final validation loss (delta {d:+.4f})"
            else:
                verdict = f"cost a larger amount of final validation loss (delta {d:+.4f})"
            lines.append(
                f"- Relative to the fixed Top-K 20% baseline, the adaptive arm {verdict} while transmitting "
                f"{_comp(summary, 'adaptive_uplink_savings_pct_vs_topk20', 1, '%')} fewer uplink bytes."
            )
        lines.append(
            "- The controller's stability test reads the loss values attached to worker pushes. Those values "
            "interleave different workers and minibatch noise, so a stricter `variability_tol` can classify normal "
            "transformer training noise as instability and step `rho` back toward the fixed baseline."
        )
        rho_initial_v = _arm_mean(summary, "async_adaptive", "rho_initial") or 0.0
        rho_final_v = _arm_mean(summary, "async_adaptive", "rho_final") or 0.0
        smax_initial_v = float(config.get("s_max") or 0)
        smax_final_v = _arm_mean(summary, "async_adaptive", "smax_final") or float(smax_initial_v)
        smax_verb = (
            "raised"
            if smax_final_v > smax_initial_v
            else ("lowered" if smax_final_v < smax_initial_v else "held")
        )
        lines.append(
            f"- The controller moved `rho` from {rho_initial_v:.3f} to {rho_final_v:.3f} and {smax_verb} "
            f"`S_max` from {smax_initial_v:.0f} to {smax_final_v:.0f}."
        )
        if (_arm_mean(summary, "async_adaptive", "rho_mean") or 0.0) > 0.16:
            lines.append(
                "- The controller spent much of the run near the fixed 20% ceiling: windows it classified as "
                "unstable pulled `rho` back up, so the byte saving stayed close to the fixed-Top-K baseline. "
                "Raising `variability_tol` (for example to 0.15-0.25) or feeding smoothed validation loss makes "
                "the stability test less noise-sensitive and lets `rho` settle lower."
            )
        adapt_total = _arm_mean(summary, "async_adaptive", "total_wire_bytes_train")
        adapt_fetch = _arm_mean(summary, "async_adaptive", "fetch_bytes_train")
        if adapt_total:
            lines.append(
                f"- Downlink delta broadcasts were {100.0 * adapt_fetch / adapt_total:.1f}% of the adaptive arm's "
                "total wire bytes, so an uplink-only saving becomes a smaller total-wire saving."
            )
    else:
        lines.append("- The adaptive arm did not complete, so no controller trajectory is reported.")
    lines += [
        "",
        "## What this says about transformers",
        "",
        "- The GPT transformer trained end-to-end through `DistributedModel` and `DistributedOptimizer`; no transformer-specific assumptions were needed in ASGC.",
        "- Gradient bucketing, Top-K sparsification, error feedback, and server-side parameter synchronization all operated on attention and MLP parameters without modification. Residual rollback is implemented and unit-tested in `torch_asgc`, but it did not fire in this run: rejection was 0.0% in three arms and 0.5% in the adaptive arm.",
        "- At equal accepted-update budgets, compression changed the bandwidth profile substantially; the quality effect is measurable and should be reported as a trade-off rather than hidden.",
        "- The adaptive controller is most valuable when the uplink is the constrained resource. With a large downlink (INT8 delta broadcast here), total-wire savings are smaller than uplink savings.",
        "",
        "## What this says about the CUDA path",
        "",
        f"- Basic CUDA preflight: `{basic}`.",
        f"- Full-stack CUDA preflight (Top-K + EF + push/fetch): `{stack}`.",
        "- Only worker 0 owns the single GPU in this configuration; the parameter server and the other workers run on CPU.",
        "- The CPU workers are intentionally slower stragglers, so the run exercises bounded staleness and downweighting. A multi-GPU slice would allow more GPU workers, but the notebook is sized for one visible device.",
        "",
        "## Caveats",
        "",
        "- Bandwidth and worker delays are simulated with deterministic profiles and per-worker RNG; wall-clock results are not measurements of a real network.",
        "- The synchronous arm is a barrier-synchronous emulation on the same asynchronous parameter-server code path. It does not average the round's gradients: after the barrier each push becomes its own server version, so this is not bulk-synchronous SGD and is not directly comparable to the research harness's averaged sync mode.",
        "- The model is a small GPT (about 1.8M parameters in the report profile), not a production LLM. Conclusions about communication savings should scale with model size, but quality sensitivity may differ.",
        "- Results are seed-dependent; with the default single seed, the table reports one run per arm. Use `ASGC_SEEDS=0,1,2` for a report-grade confidence interval.",
        "- Accuracy is next-token (teacher-forced) accuracy on held-out text, not generation quality or task accuracy. The random baseline for the 65-character vocabulary is about 1.54% top-1 and 7.69% top-5.",
        "- If the corpus source is `synthetic-fallback`, the absolute loss values are not comparable to Tiny Shakespeare runs; use the comparisons within the run.",
        "",
        "## Suggested report bullets",
        "",
        f"1. ASGC's asynchronous path was **{_comp(summary, 'async_vs_sync_speedup', unit='x')}** faster than barrier-synchronous training under the injected worker-delay profiles.",
        f"2. Top-K 20% with error feedback cut uplink traffic by **{_comp(summary, 'topk20_uplink_ratio_vs_identity', unit='x')}** while changing final loss by **{_comp(summary, 'topk20_delta_final_loss', 4)}** and top-1 accuracy by **{_pp(summary, 'topk20_delta_final_acc1')}** on a GPT transformer.",
        f"3. The adaptive controller cut uplink bytes by **{_comp(summary, 'adaptive_uplink_savings_pct_vs_topk20', 1, '%')}** and total wire bytes by **{_comp(summary, 'adaptive_total_savings_pct_vs_topk20', 1, '%')}** relative to fixed 20% Top-K, with **{_comp(summary, 'adaptive_wall_speedup_vs_topk20', unit='x')}** wall-clock.",
        f"4. The adaptive controller moved `rho` from 0.200 toward **{_fmt_value(_arm_mean(summary, 'async_adaptive', 'rho_final'), 3)}** and moved `S_max` toward **{_fmt_value(_arm_mean(summary, 'async_adaptive', 'smax_final'), 1)}**, demonstrating closed-loop adaptation under communication pressure.",
        "5. The full ASGC stack (Top-K, error feedback, residual-rollback support, parameter-server synchronization) ran on a GPT transformer with worker 0 on a CUDA device and the remaining workers on CPU. Rollback did not fire at these rejection rates.",
        "",
        "## Files in this bundle",
        "",
        "- `FINDINGS_TRANSFORMER.md` - this report.",
        "- `results.json` - aggregate summary, per-seed summaries, raw result records, and environment metadata.",
        "- `summary_per_seed.csv` and `summary_aggregate.csv` - report tables.",
        "- `eval_curves.csv`, `push_events.csv`, `monitor_samples.csv`, `worker_summary.csv` - raw telemetry.",
        "- `checkpoints/*.pt` - final server state dict per arm and seed, for checkpoint-only accuracy re-evaluation.",
        "- `plots/*.png` - wall-clock, wire bytes, loss curves, accuracy curves, final-accuracy bars, time-to-target, rejections, adaptive trajectory, and bandwidth-quality trade-off.",
        "- `config.json` and `environment.json` - reproducibility metadata.",
        "",
    ]
    return "\n".join(lines)


def write_readme(
    output_dir: Path,
    config: ExpConfig,
    environment: dict[str, Any],
    findings: str,
) -> None:
    lines = [
        "# ASGC Transformer Run",
        "",
        f"Profile: `{config.profile_name}`",
        f"Seeds: `{environment.get('seeds')}`",
        f"Arms: `{[arm.name for arm in ARMS]}`",
        f"Model parameters: `{environment.get('model_params'):,}`",
        f"Target updates: `{config.target_updates}`",
        f"Uplink / downlink: `{config.bandwidth_mbps:g}` / `{config.downlink_mbps:g}` Mbps",
        f"GPU device: `{(environment.get('gpu') or {}).get('device', 'unavailable')}`",
        f"GPU worker used: `{environment.get('gpu_worker_used')}`",
        f"Started: `{environment.get('started_at')}`",
        f"Finished: `{environment.get('finished_at')}`",
        "",
        "## What to send back",
        "",
        "Send the zip bundle, or at minimum:",
        "",
        "- `results.json`",
        "- `summary_aggregate.csv`",
        "- `FINDINGS_TRANSFORMER.md`",
        "- `environment.json`",
        "- `plots/`",
        "- `checkpoints/*.pt` (optional; only needed to re-evaluate accuracy without retraining)",
        "",
        "If the CUDA preflight failed, include the exact `gpu_preflight` and `gpu_stack_preflight` errors from `environment.json`.",
        "",
        "## Notes",
        "",
        "- The report profile uses a GPT transformer, not a Hugging Face checkpoint. The library's HF/Lightning plugins are separate integration points.",
        "- Accuracy is top-1/top-5 next-token accuracy on the held-out split, not classification or generation accuracy.",
        "- Wire bytes are exact compressed-payload sizes. Dense FP32 is 4 bytes/parameter; Top-K is 8 bytes/selected entry.",
        "- The downlink uses the server's INT8 delta broadcast, so adaptive uplink compression does not reduce downlink bytes.",
        "- The synchronous arm is barrier-synchronous emulation on the same asynchronous code path.",
        "",
        "## Findings preview",
        "",
        findings.split("## Suggested report bullets")[0].strip(),
        "",
    ]
    (output_dir / "README.md").write_text("\n".join(lines), encoding="utf-8")

## Run the experiment

The cell below runs the full matrix: for each seed, build a fresh GPT template, then run every arm against a fresh parameter server. Results are saved incrementally after each arm, so an interrupted session still leaves a usable `results.json` and CSV files.

If the full-stack CUDA preflight fails, the run stops with an error because a GPU is required by default. Set `ASGC_CUDA_STRICT=0` to keep the GPU for final evaluation and run the workers on CPU instead. The failure text is preserved in `environment.json` and `FINDINGS_TRANSFORMER.md`.

In [ ]:
def git_commit(repo: Path) -> str | None:
    try:
        return subprocess.check_output(
            ["git", "-C", str(repo), "rev-parse", "HEAD"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except Exception:
        return None


def build_environment() -> dict[str, Any]:
    return {
        "notebook": "transformer_asgc_cuda.ipynb",
        "started_at": datetime.now(timezone.utc).isoformat(),
        "platform": platform.platform(),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version() if torch.cuda.is_available() else None,
        "cpu_count": os.cpu_count(),
        "repo": str(REPO_DIR),
        "repo_commit": git_commit(REPO_DIR),
        "corpus_source": CORPUS_SOURCE,
        "corpus_chars": len(TEXT),
        "model_params": MODEL_PARAMS,
        "config": asdict(CONFIG),
        "seeds": list(SEEDS),
        "arms": [asdict(arm) for arm in ARMS],
        "gpu": GPU_INFO,
        "gpu_preflight": GPU_PREFLIGHT,
        "gpu_stack_preflight": {"ok": False, "skipped": True},
        "gpu_worker_used": False,
        "profile": PROFILE,
        "output_dir": str(OUTPUT_DIR),
    }


ENVIRONMENT = build_environment()
ALL_RESULTS: list[dict[str, Any]] = []
RUN_STARTED = time.perf_counter()

print("=== CUDA stack preflight ===")
if GPU_DEVICE is not None:
    STACK_PREFLIGHT = gpu_stack_preflight(TEMPLATE, CONFIG, TRAIN_WINDOWS, GPU_DEVICE)
    ENVIRONMENT["gpu_stack_preflight"] = STACK_PREFLIGHT
    print("gpu stack preflight:", STACK_PREFLIGHT)
    if not STACK_PREFLIGHT.get("ok"):
        if CONFIG.cuda_strict:
            raise RuntimeError(f"CUDA stack preflight failed: {STACK_PREFLIGHT}")
        print(
            "CUDA stack preflight failed; keeping the GPU for final evaluation "
            "and running workers on CPU."
        )
        CONFIG.use_gpu_worker = False
    else:
        CONFIG.use_gpu_worker = True
else:
    CONFIG.use_gpu_worker = False
    CONFIG.eval_on_gpu = False

ENVIRONMENT["gpu_worker_used"] = bool(CONFIG.use_gpu_worker and GPU_DEVICE is not None)
ENVIRONMENT["config"] = asdict(CONFIG)
print(
    f"gpu worker: {CONFIG.use_gpu_worker} | "
    f"final eval on GPU: {CONFIG.eval_on_gpu and GPU_DEVICE is not None}"
)

try:
    for seed in SEEDS:
        seed_config = replace(CONFIG, seed=seed)
        template = build_template(seed_config, seed)
        print(f"\n########## seed {seed} ##########")
        for arm in ARMS:
            if TIME_BUDGET_S > 0 and (time.perf_counter() - RUN_STARTED) >= TIME_BUDGET_S:
                print(f"time budget reached; stopping before {arm.name}")
                break
            print(f"\n=== seed {seed} | {arm.name} ===")
            arm_started = time.perf_counter()
            result = run_arm(
                arm,
                seed_config,
                template,
                TOKENIZER,
                TRAIN_WINDOWS,
                VAL_BATCHES,
                GPU_DEVICE,
                OUTPUT_DIR,
                verbose=True,
            )
            ALL_RESULTS.append(result)
            if "error" in result:
                print("  ERROR:", result)
            else:
                print(
                    f"  done: wall={result['wall_s']:.1f}s "
                    f"updates={result['completed_updates']} "
                    f"loss={result['final_val_loss']:.4f} "
                    f"acc1={100.0 * result['final_acc1']:.2f}% "
                    f"acc5={100.0 * result['final_acc5']:.2f}% "
                    f"uplink={result['push_bytes']/1e6:.1f}MB "
                    f"downlink={result['fetch_bytes_train']/1e6:.1f}MB "
                    f"rho={result['rho_mean']:.3f} S_max={result['smax_mean']:.2f}"
                )
                if result["worker_errors"]:
                    print("  worker errors:", result["worker_errors"])
            partial_summary = aggregate_results(ALL_RESULTS, seed_config)
            ENVIRONMENT["elapsed_s"] = time.perf_counter() - RUN_STARTED
            save_outputs(ALL_RESULTS, partial_summary, seed_config, ENVIRONMENT, OUTPUT_DIR)
            print(f"  arm elapsed: {time.perf_counter() - arm_started:.1f}s")
        del template
        gc.collect()
        if TIME_BUDGET_S > 0 and (time.perf_counter() - RUN_STARTED) >= TIME_BUDGET_S:
            break
except KeyboardInterrupt:
    print("\nInterrupted; saving partial results.")

FINAL_SUMMARY = aggregate_results(ALL_RESULTS, CONFIG)
ENVIRONMENT["finished_at"] = datetime.now(timezone.utc).isoformat()
ENVIRONMENT["elapsed_s"] = time.perf_counter() - RUN_STARTED
ENVIRONMENT["config"] = asdict(CONFIG)
save_outputs(ALL_RESULTS, FINAL_SUMMARY, CONFIG, ENVIRONMENT, OUTPUT_DIR)
PLOT_PATHS = make_plots(ALL_RESULTS, FINAL_SUMMARY, CONFIG, OUTPUT_DIR)
FINDINGS = findings_text(ALL_RESULTS, FINAL_SUMMARY, ENVIRONMENT)
(OUTPUT_DIR / "FINDINGS_TRANSFORMER.md").write_text(FINDINGS, encoding="utf-8")
write_readme(OUTPUT_DIR, CONFIG, ENVIRONMENT, FINDINGS)

print("\n" + FINDINGS)
print(f"\nTotal elapsed: {ENVIRONMENT['elapsed_s']:.1f}s")
print("Output directory:", OUTPUT_DIR)
print("Results:", OUTPUT_DIR / "results.json")
print("Findings:", OUTPUT_DIR / "FINDINGS_TRANSFORMER.md")

## Optional: re-evaluate saved checkpoints

Every arm writes its final server state dict to `checkpoints/<arm>_seed<seed>.pt`. The cell below loads those weights into a fresh copy of the model and recomputes loss, perplexity, top-1, and top-5 accuracy on the held-out split. It does not retrain, so it is useful for adding or auditing metrics after a run.

In [ ]:
def evaluate_checkpoints(output_dir: Path, val_batches: list[tuple[torch.Tensor, torch.Tensor]]) -> list[dict[str, Any]]:
    checkpoint_dir = Path(output_dir) / "checkpoints"
    rows: list[dict[str, Any]] = []
    if not checkpoint_dir.is_dir():
        print("No checkpoints found; run the experiment cell first.")
        return rows
    for path in sorted(checkpoint_dir.glob("*.pt")):
        try:
            payload = torch.load(path, map_location="cpu", weights_only=False)
        except TypeError:
            payload = torch.load(path, map_location="cpu")
        model = copy.deepcopy(TEMPLATE)
        model.load_state_dict(payload["state_dict"])
        metrics = evaluate_cpu(model, val_batches)
        rows.append(
            {
                "checkpoint": path.name,
                "arm": payload.get("arm", path.stem),
                "seed": payload.get("seed"),
                "val_loss": metrics["loss"],
                "ppl": metrics["ppl"],
                "acc1": metrics["acc1"],
                "acc5": metrics["acc5"],
                "tokens": metrics["tokens"],
            }
        )
        del model
        gc.collect()
    return rows


CHECKPOINT_EVALS = evaluate_checkpoints(OUTPUT_DIR, VAL_BATCHES)
if CHECKPOINT_EVALS:
    with (Path(OUTPUT_DIR) / "checkpoint_eval.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(CHECKPOINT_EVALS[0]))
        writer.writeheader()
        writer.writerows(CHECKPOINT_EVALS)
    for row in CHECKPOINT_EVALS:
        print(
            f"{row['arm']:16s} seed={row['seed']} loss={row['val_loss']:.4f} "
            f"ppl={row['ppl']:.2f} acc1={100.0 * row['acc1']:.2f}% "
            f"acc5={100.0 * row['acc5']:.2f}%"
        )

## Aggregate table

The table below is rendered with pandas when available and falls back to the Markdown table otherwise. `+/-` is one standard deviation across seeds.

In [ ]:
try:
    import pandas as pd
    from IPython.display import display

    rows = []
    for arm_name, metrics in FINAL_SUMMARY["arms"].items():
        row: dict[str, Any] = {
            "arm": arm_name,
            "seeds": int(metrics.get("n_seeds", {}).get("n", 0)),
        }
        for key in [
            "wall_s",
            "completed_updates",
            "final_val_loss",
            "final_ppl",
            "final_acc1",
            "final_acc5",
            "push_bytes",
            "fetch_bytes_train",
            "total_wire_bytes_train",
            "mean_tau",
            "rejection_rate",
            "rho_mean",
            "smax_mean",
            "time_to_target_s",
        ]:
            entry = metrics.get(key)
            if entry and entry.get("mean") is not None:
                row[f"{key}_mean"] = entry["mean"]
                row[f"{key}_std"] = entry["std"]
        rows.append(row)
    display(pd.DataFrame(rows))
except Exception:
    print(format_table(FINAL_SUMMARY))

## Plots

Evaluation runs on a common accepted-update grid, so every arm has validation points at the same server versions. The saved PNGs include `loss_vs_updates.png` (all baselines in one validation-loss graph), `final_accuracy_bar.png` (final top-1/top-5 accuracy per arm), accuracy curves, wire bytes, wall-clock, time-to-target, rejections, adaptive trajectory, and the bandwidth-quality trade-off. They are also included in the zip bundle.

In [ ]:
try:
    from IPython.display import Image, display as display_image

    if not PLOT_PATHS:
        print("No plots were generated.")
    for path in PLOT_PATHS:
        print(path.name)
        display_image(Image(filename=str(path), width=900))
except Exception as exc:
    print("Inline plot display unavailable:", exc)
    print("Plot files:", [str(path) for path in PLOT_PATHS])

## Optional: ecosystem plugins

This notebook's executable path deliberately uses a pure-PyTorch GPT so that it runs without `transformers`, `datasets`, or tokenizer downloads. The library also ships integrations for Hugging Face Trainer and PyTorch Lightning:

- `ASGCTrainer` / `ASGCTrainerCallback` wrap a Hugging Face `Trainer` and report live loss telemetry.
- `ASGCStrategy` is a Lightning strategy that wraps the model and optimizers with ASGC.

The cell below only checks that the plugin modules import; it does not download a model or run an HF job.

In [ ]:
print("Optional ecosystem plugins:")
try:
    from torch_asgc.plugins import (
        ASGCTrainer,
        ASGCTrainerCallback,
        ASGCStrategy,
        wrap_hf_trainer,
    )

    print("  imports OK:", ASGCTrainer.__name__, ASGCTrainerCallback.__name__)
    print("  Lightning strategy:", ASGCStrategy.__name__)
    print("  helper:", wrap_hf_trainer.__name__)
except Exception as exc:
    print("  plugin import skipped:", type(exc).__name__, exc)

## Bundle and download

The cell below creates a timestamped zip with everything needed to reproduce and report the run. On Colab it also triggers a browser download (disable with `ASGC_AUTO_DOWNLOAD=0`). On Kaggle the zip is written next to the results directory and persists as notebook output.

In [ ]:
def make_bundle(output_dir: Path) -> Path:
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    bundle = output_dir.parent / f"{output_dir.name}_{stamp}.zip"
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(output_dir.rglob("*")):
            if path.is_file() and path != bundle:
                archive.write(path, path.relative_to(output_dir.parent))
    return bundle


BUNDLE_PATH = make_bundle(OUTPUT_DIR)
print(f"bundle: {BUNDLE_PATH} ({BUNDLE_PATH.stat().st_size / 1e6:.2f} MB)")
print("contents:")
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    for info in archive.infolist():
        print(f"  {info.filename} ({info.file_size / 1e3:.1f} kB)")

downloaded = False
try:
    from google.colab import files as colab_files

    if AUTO_DOWNLOAD:
        colab_files.download(str(BUNDLE_PATH))
        downloaded = True
except Exception:
    pass
if not downloaded:
    print("\nIf the download did not start, open the file browser and download:")
    print(BUNDLE_PATH)

## What to send back for the report

Send the zip, or at minimum these files:

- `results.json`
- `summary_aggregate.csv`
- `FINDINGS_TRANSFORMER.md`
- `environment.json`
- `plots/`
- `checkpoints/` (optional; lets the accuracy be re-evaluated without retraining)

The notebook's findings file already contains the arm table, headline comparisons, accuracy, adaptive-controller trajectory, CUDA preflight status, caveats, and ready-to-paste report bullets. The `results.json` file preserves every per-seed record and the raw telemetry files are included for audit. Accuracy is **next-token top-1/top-5 accuracy** on held-out text, not classification accuracy.

### Mapping to the existing findings

- **F1 (asynchrony speedup):** `async_identity` vs `sync_identity` wall-clock.
- **F2/F5 (Top-K + EF dominance):** `async_topk20` vs `async_identity` uplink ratio, loss delta, and top-1 accuracy delta.
- **F6 (error-feedback recovery):** not isolated in this default matrix, because both Top-K arms use error feedback. The existing CIFAR-10 experiment covers the EF-vs-no-EF comparison; add an uncompensated Top-K arm only if the transformer run needs its own F6 number.
- **F8/F9 (bandwidth-bottleneck speedup):** time-to-target under the simulated uplink constraint.
- **F11 (adaptive byte savings):** `adaptive_uplink_savings_pct_vs_topk20` and `adaptive_total_savings_pct_vs_topk20`.

### Troubleshooting

- **CUDA unavailable:** the notebook stops because a GPU is required by default. Ensure the runtime has a GPU and that the installed PyTorch is a CUDA build (`torch.cuda.is_available()` must be `True`; check `nvidia-smi` and `torch.version.cuda`). Install a CUDA-enabled torch wheel, restart the kernel, and rerun. Set `ASGC_DISABLE_CUDA=1` for a CPU-only smoke test. The notebook does not install packages automatically.
- **Basic CUDA preflight fails:** check that the GPU is visible, that the selected device index is valid (`ASGC_CUDA_DEVICE`), and that there is enough free GPU memory. The error text is in `environment.json`.
- **Stack preflight fails:** the GPU worker path hit an unsupported op in compression/bucketing or ran out of device memory. The run stops by default. Set `ASGC_CUDA_STRICT=0` to keep the GPU for final evaluation and move workers to CPU. Send the exact error; it is the most useful signal for the library port.
- **Corpus download fails:** the synthetic fallback keeps the experiment runnable, but absolute losses are not comparable to Tiny Shakespeare. Re-run with network access for report numbers.
- **Plots are missing:** install `matplotlib`/`numpy`; all numbers are still in the CSV/JSON outputs.
- **Run is too long:** use `ASGC_NOTEBOOK_PROFILE=quick`, one seed (`ASGC_SEEDS=0`), fewer arms (`ASGC_ARMS=async_identity,async_topk20,async_adaptive`), or a smaller `ASGC_TARGET`.